# Eulerian Video Magnification — Real vs. AI-Generated Video
### Experimental probe: Can EVM detect the absence of biological signal in synthetic video?

**Background:** MIT CSAIL's 2012 Eulerian Video Magnification (Wu et al., SIGGRAPH 2012) amplifies
imperceptible temporal variations in video — blood flow in faces, micro-vibrations from sound.
This notebook applies EVM to both real human video and AI-generated/deepfake video of the same subject
to compare the amplified signals.

**Hypothesis:** Real video will show periodic color variation in the heartbeat frequency band (0.8–3 Hz).
AI-generated video will show noise or structurally different patterns in the same band.

**Important caveat:** Face-swap deepfakes are built on *real* video, so the underlying motion signal
may be partially preserved — this is itself an interesting finding.

---

## Step 1 — Install dependencies

In [ ]:
!pip install opencv-python-headless scipy numpy matplotlib imageio --quiet
print('Dependencies installed.')

## Step 2 — Core EVM implementation
Pure Python/NumPy implementation of Eulerian Video Magnification (color mode).
Based on Wu et al. 2012, using Laplacian pyramid + ideal bandpass filter.

In [ ]:
import cv2
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import os
import warnings
warnings.filterwarnings('ignore')

# ── Pyramid utilities ──────────────────────────────────────────────────────────

def build_gaussian_pyramid(frame, levels):
    """Build a Gaussian pyramid for a single frame."""
    pyramid = [frame.copy()]
    for _ in range(levels - 1):
        frame = cv2.pyrDown(frame)
        pyramid.append(frame)
    return pyramid

def build_laplacian_pyramid(frame, levels):
    """Build a Laplacian pyramid for a single frame."""
    gaussian_pyr = build_gaussian_pyramid(frame, levels)
    laplacian_pyr = []
    for i in range(levels - 1):
        size = (gaussian_pyr[i].shape[1], gaussian_pyr[i].shape[0])
        expanded = cv2.pyrUp(gaussian_pyr[i + 1], dstsize=size)
        laplacian = cv2.subtract(gaussian_pyr[i], expanded)
        laplacian_pyr.append(laplacian)
    laplacian_pyr.append(gaussian_pyr[-1])  # coarsest level
    return laplacian_pyr

# ── Temporal filtering ─────────────────────────────────────────────────────────

def ideal_bandpass_filter(signal_stack, fps, freq_low, freq_high):
    """
    Apply ideal bandpass filter in the frequency domain.
    signal_stack: (T, H, W, C) array
    """
    T = signal_stack.shape[0]
    freqs = np.fft.fftfreq(T, d=1.0 / fps)
    fft = np.fft.fft(signal_stack, axis=0)
    mask = (np.abs(freqs) >= freq_low) & (np.abs(freqs) <= freq_high)
    fft[~mask] = 0
    filtered = np.fft.ifft(fft, axis=0).real
    return filtered

def butter_bandpass_filter(signal_stack, fps, freq_low, freq_high, order=5):
    """
    Butterworth bandpass filter — smoother than ideal, better for motion.
    signal_stack: (T, H, W, C) array
    """
    nyq = fps / 2.0
    low = freq_low / nyq
    high = min(freq_high / nyq, 0.99)
    b, a = scipy.signal.butter(order, [low, high], btype='band')
    # Apply along time axis (axis=0)
    orig_shape = signal_stack.shape
    flat = signal_stack.reshape(orig_shape[0], -1)
    filtered_flat = scipy.signal.filtfilt(b, a, flat, axis=0)
    return filtered_flat.reshape(orig_shape)

# ── Main EVM function ──────────────────────────────────────────────────────────

def evm_color(frames, fps, freq_low=0.83, freq_high=3.0,
              amplification=50, pyramid_levels=4,
              chrom_attenuation=0.1, filter_type='ideal'):
    """
    Eulerian Video Magnification — color mode.
    Amplifies temporal color variations in the freq_low–freq_high Hz band.

    Args:
        frames: list of RGB float32 frames in [0,1]
        fps: video frame rate
        freq_low: lower bound of bandpass (Hz)  — 0.83 ≈ 50 BPM
        freq_high: upper bound of bandpass (Hz) — 3.0  ≈ 180 BPM
        amplification: how much to amplify the filtered signal
        pyramid_levels: Gaussian pyramid depth
        chrom_attenuation: reduce chrominance amplification to limit artifacts
        filter_type: 'ideal' or 'butter'

    Returns:
        magnified_frames: list of magnified RGB float32 frames
        filtered_signal: the raw filtered pyramid signal (for analysis)
    """
    T = len(frames)
    print(f'  Processing {T} frames at {fps:.1f} fps...')

    # Convert to YIQ-like color space (separate luminance from chrominance)
    # We use a simple RGB-to-YIQ approximation
    def rgb_to_yiq(frame):
        r, g, b = frame[:,:,0], frame[:,:,1], frame[:,:,2]
        y = 0.299*r + 0.587*g + 0.114*b
        i = 0.596*r - 0.274*g - 0.322*b
        q = 0.211*r - 0.523*g + 0.312*b
        return np.stack([y, i, q], axis=-1)

    def yiq_to_rgb(frame):
        y, i, q = frame[:,:,0], frame[:,:,1], frame[:,:,2]
        r = y + 0.956*i + 0.621*q
        g = y - 0.272*i - 0.647*q
        b = y - 1.106*i + 1.703*q
        return np.stack([r, g, b], axis=-1)

    # Convert all frames
    yiq_frames = [rgb_to_yiq(f.astype(np.float32)) for f in frames]

    # Build Gaussian pyramid for each frame, collect lowest level
    # (color magnification uses the coarsest Gaussian level)
    pyramid_stack = []
    for frame in yiq_frames:
        pyr = build_gaussian_pyramid(frame, pyramid_levels)
        pyramid_stack.append(pyr[-1])  # coarsest level

    pyramid_stack = np.array(pyramid_stack)  # (T, h, w, 3)
    print(f'  Pyramid level shape: {pyramid_stack.shape}')

    # Temporal bandpass filter
    if filter_type == 'ideal':
        filtered = ideal_bandpass_filter(pyramid_stack, fps, freq_low, freq_high)
    else:
        filtered = butter_bandpass_filter(pyramid_stack, fps, freq_low, freq_high)

    # Amplify — attenuate chrominance channels to reduce artifacts
    amplified = filtered.copy()
    amplified[:, :, :, 0] *= amplification          # Y (luminance)
    amplified[:, :, :, 1] *= amplification * chrom_attenuation  # I
    amplified[:, :, :, 2] *= amplification * chrom_attenuation  # Q

    # Reconstruct: upsample amplified signal back and add to original
    magnified_frames = []
    for i, orig_yiq in enumerate(yiq_frames):
        h, w = orig_yiq.shape[:2]
        # Upsample the amplified low-freq signal to original size
        amp_up = cv2.resize(amplified[i], (w, h), interpolation=cv2.INTER_LINEAR)
        magnified_yiq = orig_yiq + amp_up
        magnified_rgb = yiq_to_rgb(magnified_yiq)
        magnified_rgb = np.clip(magnified_rgb, 0, 1)
        magnified_frames.append(magnified_rgb)

    return magnified_frames, filtered

print('EVM core loaded.')

## Step 3 — Video I/O utilities

In [ ]:
def load_video(path, max_frames=None, target_fps=None, resize=None):
    """
    Load video frames as float32 RGB arrays in [0,1].

    Args:
        path: path to video file
        max_frames: limit number of frames loaded
        target_fps: if set, subsample to this fps
        resize: (width, height) tuple to resize frames

    Returns:
        frames: list of float32 RGB frames
        fps: original video fps
    """
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {path}')

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'  Video: {total} frames @ {fps:.1f} fps')

    # Calculate frame sampling interval if resampling fps
    sample_interval = 1
    effective_fps = fps
    if target_fps and target_fps < fps:
        sample_interval = int(round(fps / target_fps))
        effective_fps = fps / sample_interval
        print(f'  Subsampling every {sample_interval} frames → effective {effective_fps:.1f} fps')

    frames = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_interval == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if resize:
                frame_rgb = cv2.resize(frame_rgb, resize)
            frames.append(frame_rgb.astype(np.float32) / 255.0)
            if max_frames and len(frames) >= max_frames:
                break
        frame_idx += 1

    cap.release()
    print(f'  Loaded {len(frames)} frames')
    return frames, effective_fps


def save_video(frames, output_path, fps=30):
    """Save list of float32 RGB frames as an MP4."""
    if not frames:
        raise ValueError('No frames to save')
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for frame in frames:
        bgr = cv2.cvtColor((frame * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        writer.write(bgr)
    writer.release()
    print(f'  Saved: {output_path}')


print('Video I/O utilities loaded.')

## Step 4 — Signal analysis utilities
These extract and compare the temporal color signal from a region of interest (face/skin area).

In [ ]:
def extract_roi_signal(frames, roi=None):
    """
    Extract mean color signal over time from a region of interest.

    Args:
        frames: list of float32 RGB frames
        roi: (x, y, w, h) tuple, or None for full frame

    Returns:
        signal: (T, 3) array of mean RGB values over time
    """
    signal = []
    for frame in frames:
        if roi:
            x, y, w, h = roi
            patch = frame[y:y+h, x:x+w]
        else:
            patch = frame
        signal.append(patch.mean(axis=(0, 1)))
    return np.array(signal)


def compute_power_spectrum(signal, fps):
    """
    Compute power spectral density of a 1D signal.
    Returns frequencies and power.
    """
    T = len(signal)
    freqs = np.fft.rfftfreq(T, d=1.0/fps)
    fft_vals = np.fft.rfft(signal - signal.mean())
    power = np.abs(fft_vals) ** 2
    return freqs, power


def detect_pulse_peak(signal, fps, freq_low=0.75, freq_high=3.5):
    """
    Detect the dominant frequency in the heartbeat band and estimate BPM.
    Returns dominant frequency (Hz), estimated BPM, and peak power.
    """
    freqs, power = compute_power_spectrum(signal, fps)
    mask = (freqs >= freq_low) & (freqs <= freq_high)
    if not mask.any():
        return None, None, 0
    band_freqs = freqs[mask]
    band_power = power[mask]
    peak_idx = np.argmax(band_power)
    dominant_freq = band_freqs[peak_idx]
    bpm = dominant_freq * 60
    peak_power = band_power[peak_idx]
    # Signal-to-noise: ratio of peak power to total power outside band
    total_power = power.sum()
    snr = peak_power / (total_power - band_power.sum() + 1e-10)
    return dominant_freq, bpm, snr


print('Signal analysis utilities loaded.')

## Step 5 — Upload your videos

Upload two video files:
- **real_video**: A real human face video (ideally frontal, relatively still, 10–30 seconds)
- **ai_video**: An AI-generated or deepfake video of comparable length and framing

Good test sources:
- **FaceForensics++** dataset (requires registration): https://github.com/ondyari/FaceForensics
- **Celeb-DF**: https://github.com/yuezunli/celeb-deepfakeforensics
- **DFDC Preview**: https://ai.facebook.com/datasets/dfdc/
- Or any known real/synthetic pair you have access to

Minimum requirements: 15+ fps, frontal face, 10+ seconds.

In [ ]:
from google.colab import files

print('Upload your REAL video:')
uploaded_real = files.upload()
real_video_path = list(uploaded_real.keys())[0]
print(f'Real video: {real_video_path}')

In [ ]:
print('Upload your AI-GENERATED / DEEPFAKE video:')
uploaded_ai = files.upload()
ai_video_path = list(uploaded_ai.keys())[0]
print(f'AI video: {ai_video_path}')

## Step 6 — Configure EVM parameters

Adjust these based on your video's frame rate and what you want to detect.

In [ ]:
# ── EVM Parameters ─────────────────────────────────────────────────────────────

# Heartbeat band: 0.83–2.5 Hz ≈ 50–150 BPM
FREQ_LOW  = 0.83   # Hz  (lower bound of heartbeat band)
FREQ_HIGH = 2.5    # Hz  (upper bound)

# How much to amplify the filtered signal
# Higher = more visible but more noise
AMPLIFICATION = 50

# Pyramid depth — 4 is standard for color magnification
PYRAMID_LEVELS = 4

# Chrominance attenuation (0–1) — reduces color artifacts
CHROM_ATTENUATION = 0.1

# Max frames to process (keep ≤300 for speed in Colab)
MAX_FRAMES = 150

# Resize frames for processing (smaller = faster)
RESIZE = (320, 240)  # (width, height)

# Target FPS for processing (subsamples if original is higher)
# EVM needs at least 2x the highest freq you want to detect
# For 2.5 Hz, you need at least 5 fps — 15 fps is comfortable
TARGET_FPS = 15

print('Parameters configured.')
print(f'  Heartbeat band: {FREQ_LOW}–{FREQ_HIGH} Hz ({FREQ_LOW*60:.0f}–{FREQ_HIGH*60:.0f} BPM)')
print(f'  Amplification: {AMPLIFICATION}x')
print(f'  Max frames: {MAX_FRAMES}')

## Step 7 — Run EVM on both videos

In [ ]:
print('Loading REAL video...')
real_frames, real_fps = load_video(
    real_video_path,
    max_frames=MAX_FRAMES,
    target_fps=TARGET_FPS,
    resize=RESIZE
)

print('\nRunning EVM on REAL video...')
real_magnified, real_filtered = evm_color(
    real_frames, real_fps,
    freq_low=FREQ_LOW,
    freq_high=FREQ_HIGH,
    amplification=AMPLIFICATION,
    pyramid_levels=PYRAMID_LEVELS,
    chrom_attenuation=CHROM_ATTENUATION
)
print('Real video EVM complete.')

In [ ]:
print('Loading AI video...')
ai_frames, ai_fps = load_video(
    ai_video_path,
    max_frames=MAX_FRAMES,
    target_fps=TARGET_FPS,
    resize=RESIZE
)

print('\nRunning EVM on AI video...')
ai_magnified, ai_filtered = evm_color(
    ai_frames, ai_fps,
    freq_low=FREQ_LOW,
    freq_high=FREQ_HIGH,
    amplification=AMPLIFICATION,
    pyramid_levels=PYRAMID_LEVELS,
    chrom_attenuation=CHROM_ATTENUATION
)
print('AI video EVM complete.')

## Step 8 — Visual comparison: frame strips

In [ ]:
def show_frame_strip(frames_orig, frames_mag, title, n_frames=6):
    """Show a strip of original vs magnified frames side by side."""
    T = len(frames_orig)
    indices = np.linspace(0, T-1, n_frames, dtype=int)

    fig, axes = plt.subplots(2, n_frames, figsize=(3*n_frames, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)

    for col, idx in enumerate(indices):
        axes[0, col].imshow(frames_orig[idx])
        axes[0, col].set_title(f't={idx}', fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(frames_mag[idx])
        axes[1, col].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=9)
    axes[1, 0].set_ylabel('EVM Magnified', fontsize=9)
    plt.tight_layout()
    plt.show()


show_frame_strip(real_frames, real_magnified, 'REAL VIDEO — Original vs EVM Magnified')
show_frame_strip(ai_frames, ai_magnified, 'AI VIDEO — Original vs EVM Magnified')

## Step 9 — Temporal color signal comparison

This is the key diagnostic: plot the mean color signal over time from the face region.
A real face should show periodic oscillation; AI-generated video should show noise or flat signal.

In [ ]:
# Extract temporal signal from center crop (approximate face region)
# Adjust ROI (x, y, w, h) to better match the face in your videos
H, W = real_frames[0].shape[:2]
# Default: center third of frame
ROI = (W//4, H//4, W//2, H//2)

real_signal = extract_roi_signal(real_frames, roi=ROI)
ai_signal   = extract_roi_signal(ai_frames, roi=ROI)

# Also extract from magnified frames to see amplified signal
real_mag_signal = extract_roi_signal(real_magnified, roi=ROI)
ai_mag_signal   = extract_roi_signal(ai_magnified, roi=ROI)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle('Temporal Color Signal — ROI Mean over Time', fontsize=13, fontweight='bold')

t_real = np.arange(len(real_frames)) / real_fps
t_ai   = np.arange(len(ai_frames)) / ai_fps

colors = ['#e63946', '#2a9d8f', '#264653']
labels = ['R', 'G', 'B']

for ch, (c, lbl) in enumerate(zip(colors, labels)):
    axes[0,0].plot(t_real, real_signal[:,ch] - real_signal[:,ch].mean(), color=c, alpha=0.8, label=lbl, linewidth=0.8)
    axes[0,1].plot(t_ai,   ai_signal[:,ch]   - ai_signal[:,ch].mean(),   color=c, alpha=0.8, label=lbl, linewidth=0.8)
    axes[1,0].plot(t_real, real_mag_signal[:,ch] - real_mag_signal[:,ch].mean(), color=c, alpha=0.8, label=lbl, linewidth=0.8)
    axes[1,1].plot(t_ai,   ai_mag_signal[:,ch]   - ai_mag_signal[:,ch].mean(),   color=c, alpha=0.8, label=lbl, linewidth=0.8)

titles = [
    'REAL — Raw Signal',
    'AI — Raw Signal',
    'REAL — After EVM Magnification',
    'AI — After EVM Magnification',
]
for ax, title in zip(axes.flat, titles):
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ΔColor (mean-subtracted)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 10 — Power spectrum comparison

Plot the frequency content of both signals. A real face should show a clear peak
in the 0.83–2.5 Hz band (heartbeat). AI video should lack this structured peak.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Power Spectrum — Heartbeat Band Analysis', fontsize=13, fontweight='bold')

for ax, signal, fps, title, color in [
    (axes[0], real_signal, real_fps, 'REAL VIDEO', '#e63946'),
    (axes[1], ai_signal,   ai_fps,   'AI VIDEO',   '#457b9d'),
]:
    # Use green channel (most sensitive to blood flow)
    freqs, power = compute_power_spectrum(signal[:, 1], fps)

    ax.semilogy(freqs, power, color=color, linewidth=1.2, alpha=0.9)

    # Shade heartbeat band
    band_mask = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)
    ax.fill_between(freqs, power, where=band_mask, alpha=0.25, color='gold',
                    label=f'Heartbeat band ({FREQ_LOW}–{FREQ_HIGH} Hz)')

    # Detect and annotate pulse peak
    dom_freq, bpm, snr = detect_pulse_peak(signal[:, 1], fps, FREQ_LOW, FREQ_HIGH)
    if dom_freq:
        band_freqs, band_power = freqs[band_mask], power[band_mask]
        peak_power = band_power.max()
        ax.axvline(dom_freq, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
        ax.annotate(f'{bpm:.0f} BPM\nSNR: {snr:.2f}',
                    xy=(dom_freq, peak_power),
                    xytext=(dom_freq + 0.2, peak_power * 2),
                    fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='black', lw=0.8))

    ax.set_xlim(0, 5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Power (log scale)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
print('\n── Pulse Detection Summary ──────────────────────────────────')
for label, signal, fps in [('REAL', real_signal, real_fps), ('AI', ai_signal, ai_fps)]:
    df, bpm, snr = detect_pulse_peak(signal[:, 1], fps, FREQ_LOW, FREQ_HIGH)
    if df:
        print(f'{label:6s} | Dominant freq: {df:.2f} Hz | Est. BPM: {bpm:.0f} | SNR: {snr:.4f}')
    else:
        print(f'{label:6s} | No clear peak detected in heartbeat band')

## Step 11 — Spatiotemporal heatmap

Visualize WHERE in the frame the periodic signal is strongest.
On real video, this should cluster over skin/face areas.
On AI video, the pattern should be structurally different — diffuse, uniform, or absent.

In [ ]:
def compute_spatiotemporal_power_map(frames, fps, freq_low, freq_high):
    """
    Compute per-pixel power in the heartbeat band.
    Returns a 2D map showing where periodic signal is strongest.
    """
    # Stack green channel: (T, H, W)
    green_stack = np.array([f[:,:,1] for f in frames])
    T, H, W = green_stack.shape

    # FFT along time axis
    freqs = np.fft.rfftfreq(T, d=1.0/fps)
    fft_stack = np.fft.rfft(green_stack - green_stack.mean(axis=0), axis=0)

    # Sum power in heartbeat band
    band_mask = (freqs >= freq_low) & (freqs <= freq_high)
    band_power = (np.abs(fft_stack[band_mask]) ** 2).sum(axis=0)
    total_power = (np.abs(fft_stack) ** 2).sum(axis=0) + 1e-10

    # Relative band power map
    rel_power = band_power / total_power
    return rel_power


print('Computing spatial power maps...')
real_power_map = compute_spatiotemporal_power_map(real_frames, real_fps, FREQ_LOW, FREQ_HIGH)
ai_power_map   = compute_spatiotemporal_power_map(ai_frames,   ai_fps,   FREQ_LOW, FREQ_HIGH)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'Spatial Heartbeat Power Map — {FREQ_LOW}–{FREQ_HIGH} Hz Band\n'
             '(Bright = strong periodic signal in heartbeat band)',
             fontsize=12, fontweight='bold')

# Reference frames
mid_real = len(real_frames) // 2
mid_ai   = len(ai_frames) // 2

axes[0].imshow(real_frames[mid_real])
axes[0].set_title('REAL — Reference Frame', fontsize=10)
axes[0].axis('off')

im1 = axes[1].imshow(real_power_map, cmap='inferno', vmin=0, vmax=real_power_map.max())
axes[1].set_title('REAL — Heartbeat Power Map', fontsize=10)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='Relative band power')

im2 = axes[2].imshow(ai_power_map, cmap='inferno',
                      vmin=0, vmax=real_power_map.max())  # same scale for comparison
axes[2].set_title('AI — Heartbeat Power Map', fontsize=10)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, label='Relative band power')

plt.tight_layout()
plt.show()

print(f'\nMax heartbeat-band power:')
print(f'  REAL: {real_power_map.max():.4f}  |  Mean: {real_power_map.mean():.4f}')
print(f'  AI:   {ai_power_map.max():.4f}  |  Mean: {ai_power_map.mean():.4f}')

## Step 12 — Save magnified videos for side-by-side review

In [ ]:
print('Saving magnified videos...')
save_video(real_magnified, 'real_evm_magnified.mp4', fps=real_fps)
save_video(ai_magnified,   'ai_evm_magnified.mp4',   fps=ai_fps)

# Download
from google.colab import files
files.download('real_evm_magnified.mp4')
files.download('ai_evm_magnified.mp4')
print('Download started.')

## Step 13 — Summary statistics

In [ ]:
print('═' * 55)
print('  EVM ANALYSIS SUMMARY')
print('═' * 55)

for label, frames, fps, signal, power_map in [
    ('REAL', real_frames, real_fps, real_signal, real_power_map),
    ('AI',   ai_frames,   ai_fps,   ai_signal,   ai_power_map),
]:
    df, bpm, snr = detect_pulse_peak(signal[:, 1], fps, FREQ_LOW, FREQ_HIGH)
    # Amplitude of raw green channel oscillation
    g = signal[:, 1]
    g_detrended = g - np.polyval(np.polyfit(np.arange(len(g)), g, 1), np.arange(len(g)))
    amplitude = g_detrended.std()

    print(f'\n  {label} VIDEO')
    print(f'    Frames processed : {len(frames)}')
    print(f'    Effective FPS    : {fps:.1f}')
    print(f'    Duration         : {len(frames)/fps:.1f}s')
    if df:
        print(f'    Dominant freq    : {df:.3f} Hz')
        print(f'    Estimated BPM    : {bpm:.0f}')
        print(f'    Band SNR         : {snr:.4f}')
    else:
        print(f'    No clear pulse peak detected')
    print(f'    Signal amplitude : {amplitude:.5f}')
    print(f'    Spatial max power: {power_map.max():.4f}')
    print(f'    Spatial mean pwr : {power_map.mean():.4f}')

print('\n' + '═' * 55)
print('  INTERPRETATION NOTES')
print('═' * 55)
print("""
  Real video with blood flow should show:
    - A clear spectral peak in the 50–150 BPM range
    - Higher SNR in the heartbeat band
    - Spatially structured power map (concentrated on skin)
    - Visible periodic color oscillation in temporal signal

  AI-generated / deepfake video may show:
    - Diffuse or absent spectral peak in heartbeat band
    - Lower or structurally different SNR
    - Uniform or noise-like spatial power map
    - Flat or aperiodic temporal signal

  NOTE: Face-swap deepfakes on real video may partially
  preserve the blood flow signal from the source actor —
  this is itself an interesting finding worth documenting.

  Fully synthetic (e.g. GAN/diffusion video) should show
  the starkest contrast with real video.
""")

---
## References

- Wu, H-Y., Rubinstein, M., Shih, E., Guttag, J., Durand, F., & Freeman, W. (2012). **Eulerian Video Magnification for Revealing Subtle Changes in the World.** *ACM Transactions on Graphics (SIGGRAPH)*, 31(4). https://people.csail.mit.edu/mrub/evm/

- Davis, A., Rubinstein, M., Wadhwa, N., Mysore, G., Durand, F., & Freeman, W. (2014). **The Visual Microphone: Passive Recovery of Sound from Video.** *ACM Transactions on Graphics (SIGGRAPH)*, 33(4). https://people.csail.mit.edu/mrub/VisualMic/

- FaceForensics++ dataset: https://github.com/ondyari/FaceForensics

---
*Notebook built for exploratory research into EVM as a probe for biological signal in real vs. synthetic video.*

---
## AXIS 2 — Physical Plausibility Envelope
### Spectral & Sensor Forensics: Did this content ever exist as light hitting a sensor?

**The core idea:** Real cameras capture light governed by physics — lens optics, Bayer filter
demosaicing, sensor noise (PRNU), chromatic aberration. These leave measurable traces in every
real image. AI-generated content approximates the *appearance* of these traces but is not
constrained by physical reality, producing characteristic signatures:

- **Noise floor:** Real sensors have structured PRNU noise; AI images have unnaturally low or
  uniformly random noise
- **Chromatic aberration:** Real lenses produce color channel misalignment at edges; AI images
  often lack this or produce it incorrectly
- **Color channel statistics:** Real camera Bayer patterns create specific inter-channel
  correlations; AI synthesis breaks these
- **Frequency spectrum:** CNNs and diffusion models leave characteristic spectral artifacts —
  periodic grid patterns in Fourier space that no real camera produces
- **Subsurface scattering:** Real skin has specific RGB ratio signatures from light penetrating
  dermis; AI skin approximates surface appearance only

This axis asks not *"was this made by AI?"* but *"does this content bear physical traces of
having existed in the world?"* — a more fundamental question for SLURRY's purposes.

---

### Step A — Noise Residual Analysis (PRNU proxy)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import ndimage, stats
from scipy.fft import fft2, fftshift
import warnings
warnings.filterwarnings('ignore')

# ── Noise extraction ──────────────────────────────────────────────────────────

def extract_noise_residual(frame, method='wavelet_approx'):
    """
    Extract high-frequency noise residual from a frame.
    Approximates the PRNU extraction pipeline: denoise, subtract from original.

    Real cameras: structured, spatially correlated noise (PRNU)
    AI images: very low variance noise OR uniformly random noise

    Args:
        frame: float32 RGB frame in [0,1]
        method: 'gaussian' or 'wavelet_approx'

    Returns:
        residual: noise residual (H, W, 3)
        stats: dict of noise statistics
    """
    if method == 'gaussian':
        # Simple Gaussian denoising — residual = original - smoothed
        sigma = 2.0
        denoised = np.stack([
            ndimage.gaussian_filter(frame[:,:,c], sigma=sigma)
            for c in range(3)
        ], axis=-1)
    else:
        # Wavelet-approximation denoising using repeated Gaussian pyramid
        # More faithful to the Lukas et al. PRNU extraction approach
        denoised = np.zeros_like(frame)
        for c in range(3):
            ch = frame[:,:,c]
            # Build and collapse Gaussian pyramid (approximates wavelet denoising)
            down = cv2.pyrDown(ch)
            up = cv2.pyrUp(down, dstsize=(ch.shape[1], ch.shape[0]))
            denoised[:,:,c] = up

    residual = frame - denoised

    # Noise statistics
    noise_stats = {
        'variance': residual.var(),
        'std': residual.std(),
        'channel_vars': [residual[:,:,c].var() for c in range(3)],
        'kurtosis': float(stats.kurtosis(residual.flatten())),
        'spatial_correlation': float(np.corrcoef(
            residual[:,:,1].flatten(),
            np.roll(residual[:,:,1], 1, axis=1).flatten()
        )[0,1])
    }

    return residual, noise_stats


def analyze_noise_across_frames(frames, n_sample=20, method='wavelet_approx'):
    """Sample frames and aggregate noise statistics."""
    indices = np.linspace(0, len(frames)-1, min(n_sample, len(frames)), dtype=int)
    all_residuals = []
    all_stats = []
    for i in indices:
        res, st = extract_noise_residual(frames[i], method=method)
        all_residuals.append(res)
        all_stats.append(st)

    # Aggregate
    agg = {
        'mean_variance': np.mean([s['variance'] for s in all_stats]),
        'mean_std': np.mean([s['std'] for s in all_stats]),
        'variance_over_time': [s['variance'] for s in all_stats],
        'mean_kurtosis': np.mean([s['kurtosis'] for s in all_stats]),
        'mean_spatial_corr': np.mean([s['spatial_correlation'] for s in all_stats]),
        'channel_var_ratio': np.mean([
            s['channel_vars'][0] / (s['channel_vars'][1] + 1e-10)
            for s in all_stats
        ]),
    }
    # Mean residual across frames (real cameras: structured; AI: near-zero)
    mean_residual = np.mean(all_residuals, axis=0)
    agg['mean_residual'] = mean_residual
    agg['mean_residual_energy'] = mean_residual.std()

    return agg, all_stats

print('Noise residual analysis loaded.')


In [ ]:
print('Analyzing noise residuals...')
real_noise_agg, real_noise_stats = analyze_noise_across_frames(real_frames, n_sample=30)
ai_noise_agg, ai_noise_stats     = analyze_noise_across_frames(ai_frames, n_sample=30)

# ── Plot 1: Mean noise residual maps ─────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Noise Residual Analysis — PRNU Proxy\n'
             'Real cameras show structured spatial noise; AI synthesis shows flat or random noise',
             fontsize=12, fontweight='bold')

mid_real = len(real_frames) // 2
mid_ai   = len(ai_frames)   // 2

# Single frame noise residuals
res_real, _ = extract_noise_residual(real_frames[mid_real])
res_ai,   _ = extract_noise_residual(ai_frames[mid_ai])

for col, (res, label, frames_list, agg) in enumerate([
    (res_real, 'REAL',  real_frames, real_noise_agg),
    (res_ai,   'AI',    ai_frames,   ai_noise_agg),
]):
    # Original frame
    mid = len(frames_list) // 2
    axes[0, col*2].imshow(frames_list[mid])
    axes[0, col*2].set_title(f'{label} — Reference Frame', fontsize=9)
    axes[0, col*2].axis('off')

    # Single-frame noise residual (green channel, amplified)
    disp = res[:,:,1]
    vmax = max(abs(disp.min()), abs(disp.max()))
    axes[0, col*2+1].imshow(disp, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[0, col*2+1].set_title(f'{label} — Noise Residual (G)', fontsize=9)
    axes[0, col*2+1].axis('off')

    # Mean residual across frames (PRNU signature)
    mean_res = agg['mean_residual'][:,:,1]
    vmax2 = max(abs(mean_res.min()), abs(mean_res.max())) + 1e-8
    axes[1, col*2].imshow(mean_res, cmap='RdBu', vmin=-vmax2, vmax=vmax2)
    axes[1, col*2].set_title(f'{label} — Mean Residual\n(PRNU proxy; structured = real camera)',
                              fontsize=9)
    axes[1, col*2].axis('off')

    # Noise variance over time
    axes[1, col*2+1].plot(agg['variance_over_time'], color='#2a9d8f', linewidth=1.2)
    axes[1, col*2+1].set_title(f'{label} — Noise Variance over Time', fontsize=9)
    axes[1, col*2+1].set_xlabel('Frame sample')
    axes[1, col*2+1].set_ylabel('Noise variance')
    axes[1, col*2+1].grid(True, alpha=0.3)
    axes[1, col*2+1].axhline(agg['mean_variance'], color='red', linestyle='--',
                              linewidth=0.8, label=f'Mean: {agg["mean_variance"]:.5f}')
    axes[1, col*2+1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\n── Noise Statistics ──────────────────────────────────────')
for label, agg in [('REAL', real_noise_agg), ('AI', ai_noise_agg)]:
    print(f'\n  {label}')
    print(f'    Mean noise variance   : {agg["mean_variance"]:.6f}')
    print(f'    Mean noise std        : {agg["mean_std"]:.6f}')
    print(f'    Mean kurtosis         : {agg["mean_kurtosis"]:.3f}')
    print(f'    Spatial correlation   : {agg["mean_spatial_corr"]:.4f}')
    print(f'    Mean residual energy  : {agg["mean_residual_energy"]:.6f}')
    print(f'    R/G channel var ratio : {agg["channel_var_ratio"]:.4f}')
print()
print('  Interpretation:')
print('    Real cameras: variance ~0.001-0.01, structured spatial correlation')
print('    AI images: variance <0.0005 OR uniformly high with no spatial structure')
print('    Kurtosis > 3 suggests heavy-tailed non-Gaussian noise (camera-like)')


### Step B — Frequency Spectrum Analysis (Fourier fingerprinting)

In [ ]:
# ── Fourier analysis ──────────────────────────────────────────────────────────
# CNN-based generators (GANs, early diffusion) leave periodic grid artifacts
# in Fourier space — "checkerboard" patterns from upsampling convolutions.
# Real cameras have smooth, isotropic frequency distributions.
# Diffusion models are better at hiding this but still show characteristic patterns.

def compute_fft_spectrum(frame, channel=1):
    """
    Compute the 2D FFT magnitude spectrum of a single channel.
    Returns log-scaled spectrum centered at DC.
    """
    ch = frame[:,:,channel].astype(np.float32)
    # Subtract mean to remove DC dominance
    ch = ch - ch.mean()
    # Window to reduce edge artifacts
    h, w = ch.shape
    window = np.outer(np.hanning(h), np.hanning(w))
    ch_windowed = ch * window
    fft = fft2(ch_windowed)
    spectrum = np.abs(fftshift(fft))
    # Log scale
    log_spectrum = np.log1p(spectrum)
    return log_spectrum


def aggregate_fft_spectra(frames, n_sample=20, channel=1):
    """Average FFT spectra across frames for a stable fingerprint."""
    indices = np.linspace(0, len(frames)-1, min(n_sample, len(frames)), dtype=int)
    spectra = []
    for i in indices:
        spec = compute_fft_spectrum(frames[i], channel=channel)
        spectra.append(spec)
    mean_spec = np.mean(spectra, axis=0)

    # Radial profile (isotropy measure — real cameras are more isotropic)
    cy, cx = np.array(mean_spec.shape) // 2
    Y, X = np.ogrid[:mean_spec.shape[0], :mean_spec.shape[1]]
    R = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
    max_r = min(cy, cx)
    radial_profile = np.array([mean_spec[R == r].mean() for r in range(max_r)])

    # Peak detection in spectrum (artifact spikes)
    center_masked = mean_spec.copy()
    cy2, cx2 = center_masked.shape[0]//2, center_masked.shape[1]//2
    center_masked[cy2-10:cy2+10, cx2-10:cx2+10] = 0
    peak_val = center_masked.max()
    mean_val = center_masked.mean()
    peak_to_mean = peak_val / (mean_val + 1e-10)

    return mean_spec, radial_profile, peak_to_mean


print('Computing FFT spectra...')
real_spec, real_radial, real_ptm = aggregate_fft_spectra(real_frames, n_sample=30)
ai_spec,   ai_radial,   ai_ptm   = aggregate_fft_spectra(ai_frames,   n_sample=30)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Frequency Spectrum Analysis\n'
             'AI generators leave characteristic spectral artifacts; real cameras are smoother',
             fontsize=12, fontweight='bold')

for row, (spec, radial, ptm, label, color) in enumerate([
    (real_spec, real_radial, real_ptm, 'REAL', '#e63946'),
    (ai_spec,   ai_radial,   ai_ptm,   'AI',   '#457b9d'),
]):
    # 2D spectrum
    im = axes[row, 0].imshow(spec, cmap='inferno', aspect='auto')
    axes[row, 0].set_title(f'{label} — Mean 2D FFT Spectrum (log)\n'
                            f'Peak-to-mean ratio: {ptm:.2f}', fontsize=9)
    axes[row, 0].axis('off')
    plt.colorbar(im, ax=axes[row,0], fraction=0.046)

    # Radial profile
    axes[row, 1].semilogy(radial, color=color, linewidth=1.2)
    axes[row, 1].set_title(f'{label} — Radial Power Profile\n'
                            f'(Flat = isotropic = camera-like)', fontsize=9)
    axes[row, 1].set_xlabel('Spatial frequency (pixels from DC)')
    axes[row, 1].set_ylabel('Mean power (log)')
    axes[row, 1].grid(True, alpha=0.3)

    # High-frequency detail (zoom into upper-right quadrant)
    h, w = spec.shape
    quad = spec[h//2:, w//2:]
    axes[row, 2].imshow(quad, cmap='hot', aspect='auto')
    axes[row, 2].set_title(f'{label} — High-freq Quadrant\n'
                            f'(Grid patterns = GAN/CNN artifacts)', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()

print(f'\nPeak-to-mean ratio in FFT spectrum:')
print(f'  REAL : {real_ptm:.2f}  (higher = more spectral spikes)')
print(f'  AI   : {ai_ptm:.2f}')
print()
print('  Note: GAN-based deepfakes often show ratio >5 (grid artifacts)')
print('  Diffusion models are better at hiding this — ratio may be closer to real')


### Step C — Chromatic Aberration Analysis

In [ ]:
# ── Chromatic aberration ──────────────────────────────────────────────────────
# Real lenses produce lateral chromatic aberration: color channels are slightly
# misaligned, especially at image edges. This is a function of optical physics.
# AI generators approximate this poorly or inconsistently.
#
# Measurement: at high-contrast edges, measure the spatial offset between
# R, G, B channel edge positions. Real images show consistent, outward-radiating
# offset patterns. AI images show absent, random, or inward offsets.

def detect_edges(frame_channel, sigma=1.5):
    """Detect edges using Canny on a single channel."""
    uint8 = (frame_channel * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(uint8, (0,0), sigma)
    edges = cv2.Canny(blurred, 30, 100)
    return edges.astype(np.float32) / 255.0


def measure_chromatic_aberration(frame):
    """
    Measure chromatic aberration by comparing edge maps across channels.

    Real lenses: R and B channels have edges slightly offset from G (centroid).
    The offset increases toward image corners (radial pattern).
    AI images: no consistent offset pattern; edges often co-located across channels.

    Returns:
        ca_metrics: dict with aberration measurements
        edge_overlay: visualization
    """
    r_edges = detect_edges(frame[:,:,0])
    g_edges = detect_edges(frame[:,:,1])
    b_edges = detect_edges(frame[:,:,2])

    # Cross-correlation between R/G and B/G edge maps
    # High correlation = well-aligned = possible AI (no physical aberration)
    # Lower correlation = misaligned = consistent with real optics
    def norm_xcorr(a, b):
        a_norm = a - a.mean()
        b_norm = b - b.mean()
        denom = np.sqrt((a_norm**2).sum() * (b_norm**2).sum()) + 1e-10
        return float(np.sum(a_norm * b_norm) / denom)

    rg_corr = norm_xcorr(r_edges, g_edges)
    bg_corr = norm_xcorr(b_edges, g_edges)
    rb_corr = norm_xcorr(r_edges, b_edges)

    # Edge density per channel
    r_density = r_edges.mean()
    g_density = g_edges.mean()
    b_density = b_edges.mean()

    # Channel edge centroid difference (real lenses: R and B centroids offset from G)
    def edge_centroid(edge_map):
        y_coords, x_coords = np.where(edge_map > 0.5)
        if len(y_coords) == 0:
            return np.array([0.0, 0.0])
        return np.array([y_coords.mean(), x_coords.mean()])

    g_centroid = edge_centroid(g_edges)
    r_offset = np.linalg.norm(edge_centroid(r_edges) - g_centroid)
    b_offset = np.linalg.norm(edge_centroid(b_edges) - g_centroid)

    # Edge map overlay for visualization
    h, w = frame.shape[:2]
    overlay = np.zeros((h, w, 3))
    overlay[:,:,0] = r_edges  # R channel edges in red
    overlay[:,:,1] = g_edges  # G channel edges in green
    overlay[:,:,2] = b_edges  # B channel edges in blue

    metrics = {
        'rg_correlation': rg_corr,
        'bg_correlation': bg_corr,
        'rb_correlation': rb_corr,
        'mean_channel_correlation': (rg_corr + bg_corr + rb_corr) / 3,
        'r_offset_from_g': r_offset,
        'b_offset_from_g': b_offset,
        'rb_asymmetry': abs(r_offset - b_offset),
        'edge_density_variance': np.var([r_density, g_density, b_density]),
    }
    return metrics, overlay, r_edges, g_edges, b_edges


print('Analyzing chromatic aberration...')
mid_real = len(real_frames) // 2
mid_ai   = len(ai_frames)   // 2

real_ca, real_overlay, real_r, real_g, real_b = measure_chromatic_aberration(real_frames[mid_real])
ai_ca,   ai_overlay,   ai_r,   ai_g,   ai_b   = measure_chromatic_aberration(ai_frames[mid_ai])

# Aggregate across frames
def aggregate_ca(frames, n=15):
    indices = np.linspace(0, len(frames)-1, min(n, len(frames)), dtype=int)
    metrics_list = [measure_chromatic_aberration(frames[i])[0] for i in indices]
    return {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}

real_ca_agg = aggregate_ca(real_frames)
ai_ca_agg   = aggregate_ca(ai_frames)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Chromatic Aberration Analysis\n'
             'Real lenses: color channel edges physically misaligned; AI: channels often co-located',
             fontsize=12, fontweight='bold')

for row, (frames_list, overlay, r_e, g_e, b_e, label) in enumerate([
    (real_frames, real_overlay, real_r, real_g, real_b, 'REAL'),
    (ai_frames,   ai_overlay,   ai_r,   ai_g,   ai_b,   'AI'),
]):
    mid = len(frames_list) // 2
    axes[row,0].imshow(frames_list[mid])
    axes[row,0].set_title(f'{label} — Original', fontsize=9)
    axes[row,0].axis('off')

    # RGB edge overlay
    axes[row,1].imshow(np.clip(overlay, 0, 1))
    axes[row,1].set_title(f'{label} — RGB Edge Overlay\n(white=all channels, color=misaligned)',
                           fontsize=9)
    axes[row,1].axis('off')

    # G channel edges
    axes[row,2].imshow(g_e, cmap='Greens')
    axes[row,2].set_title(f'{label} — G Channel Edges', fontsize=9)
    axes[row,2].axis('off')

    # R-B difference (aberration map)
    diff = r_e - b_e
    vmax = max(abs(diff.min()), abs(diff.max())) + 1e-8
    axes[row,3].imshow(diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[row,3].set_title(f'{label} — R minus B Edge Map\n(non-zero = lateral aberration)',
                           fontsize=9)
    axes[row,3].axis('off')

plt.tight_layout()
plt.show()

print('\n── Chromatic Aberration Metrics ──────────────────────────────')
for label, ca in [('REAL', real_ca_agg), ('AI', ai_ca_agg)]:
    print(f'\n  {label}')
    print(f'    Mean channel edge correlation : {ca["mean_channel_correlation"]:.4f}')
    print(f'    R-G edge correlation          : {ca["rg_correlation"]:.4f}')
    print(f'    B-G edge correlation          : {ca["bg_correlation"]:.4f}')
    print(f'    R offset from G centroid      : {ca["r_offset_from_g"]:.2f} px')
    print(f'    B offset from G centroid      : {ca["b_offset_from_g"]:.2f} px')
    print(f'    R-B asymmetry                 : {ca["rb_asymmetry"]:.2f} px')
print()
print('  Real cameras: R-G and B-G correlations slightly <1.0; R and B offsets')
print('  consistently outward from image center; R-B asymmetry present')
print('  AI images: correlations near 1.0 (channels co-located); or inconsistent offsets')


### Step D — Color Channel Statistics & Physical Plausibility

In [ ]:
# ── Color channel statistics ──────────────────────────────────────────────────
# Real cameras have characteristic inter-channel relationships driven by:
#   - Bayer filter spectral response
#   - Sensor well capacity and gain
#   - White balance processing
#   - The physics of light interaction with matter
#
# Key signatures:
#   - Green channel carries ~2x the signal of R and B (Bayer: 2 green pixels per 4)
#   - Inter-channel correlations follow predictable curves
#   - Skin pixels have specific R/G/B ratios from subsurface scattering
#   - Saturation distribution: AI images are often oversaturated or unnaturally uniform

def analyze_color_statistics(frames, n_sample=20):
    """
    Analyze color channel statistics for physical plausibility.
    """
    indices = np.linspace(0, len(frames)-1, min(n_sample, len(frames)), dtype=int)
    all_r, all_g, all_b = [], [], []
    rg_corrs, rb_corrs, bg_corrs = [], [], []
    saturations, hue_stds = [], []

    for i in indices:
        frame = frames[i]
        r, g, b = frame[:,:,0].flatten(), frame[:,:,1].flatten(), frame[:,:,2].flatten()
        all_r.extend(r); all_g.extend(g); all_b.extend(b)

        rg_corrs.append(np.corrcoef(r, g)[0,1])
        rb_corrs.append(np.corrcoef(r, b)[0,1])
        bg_corrs.append(np.corrcoef(b, g)[0,1])

        # HSV saturation
        hsv = cv2.cvtColor((frame*255).astype(np.uint8), cv2.COLOR_RGB2HSV)
        saturations.append(hsv[:,:,1].mean() / 255.0)
        hue_stds.append(hsv[:,:,0].std())

    all_r, all_g, all_b = np.array(all_r), np.array(all_g), np.array(all_b)

    return {
        'r_mean': all_r.mean(), 'g_mean': all_g.mean(), 'b_mean': all_b.mean(),
        'r_std': all_r.std(),   'g_std': all_g.std(),   'b_std': all_b.std(),
        'g_r_ratio': all_g.mean() / (all_r.mean() + 1e-10),
        'g_b_ratio': all_g.mean() / (all_b.mean() + 1e-10),
        'rg_corr': np.mean(rg_corrs),
        'rb_corr': np.mean(rb_corrs),
        'bg_corr': np.mean(bg_corrs),
        'mean_saturation': np.mean(saturations),
        'saturation_std': np.std(saturations),
        'hue_std': np.mean(hue_stds),
        'all_r': all_r, 'all_g': all_g, 'all_b': all_b,
    }


print('Analyzing color statistics...')
real_color = analyze_color_statistics(real_frames)
ai_color   = analyze_color_statistics(ai_frames)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Color Channel Statistics — Physical Plausibility\n'
             'Real cameras follow Bayer filter physics; AI synthesis often violates these',
             fontsize=12, fontweight='bold')
gs = gridspec.GridSpec(2, 4, figure=fig)

# Sample for scatter (too many points otherwise)
n_scatter = 5000
rng = np.random.default_rng(42)

for col, (color_stats, label, c) in enumerate([
    (real_color, 'REAL', '#e63946'),
    (ai_color,   'AI',   '#457b9d'),
]):
    idx = rng.integers(0, len(color_stats['all_r']), n_scatter)
    r_s = color_stats['all_r'][idx]
    g_s = color_stats['all_g'][idx]
    b_s = color_stats['all_b'][idx]

    # RGB histograms
    ax0 = fig.add_subplot(gs[0, col*2])
    ax0.hist(r_s, bins=50, alpha=0.6, color='red',   density=True, label='R')
    ax0.hist(g_s, bins=50, alpha=0.6, color='green', density=True, label='G')
    ax0.hist(b_s, bins=50, alpha=0.6, color='blue',  density=True, label='B')
    ax0.set_title(f'{label} — Channel Distributions', fontsize=9)
    ax0.set_xlabel('Pixel value'); ax0.legend(fontsize=7)
    ax0.grid(True, alpha=0.3)

    # R vs G scatter (Bayer correlation)
    ax1 = fig.add_subplot(gs[0, col*2+1])
    ax1.scatter(r_s, g_s, alpha=0.03, s=1, color=c)
    ax1.plot([0,1],[0,1], 'k--', alpha=0.3, linewidth=0.8)
    ax1.set_title(f'{label} — R vs G\n(corr={color_stats["rg_corr"]:.3f})', fontsize=9)
    ax1.set_xlabel('R'); ax1.set_ylabel('G')
    ax1.set_xlim(0,1); ax1.set_ylim(0,1)

    # Channel means bar chart
    ax2 = fig.add_subplot(gs[1, col*2])
    channels = ['R', 'G', 'B']
    means = [color_stats['r_mean'], color_stats['g_mean'], color_stats['b_mean']]
    stds  = [color_stats['r_std'],  color_stats['g_std'],  color_stats['b_std']]
    bars = ax2.bar(channels, means, color=['red','green','blue'], alpha=0.7)
    ax2.errorbar(channels, means, yerr=stds, fmt='none', color='black', capsize=4)
    ax2.set_title(f'{label} — Channel Means\nG/R={color_stats["g_r_ratio"]:.3f}, '
                  f'G/B={color_stats["g_b_ratio"]:.3f}', fontsize=9)
    ax2.set_ylabel('Mean value'); ax2.grid(True, alpha=0.3)

    # Saturation info as text
    ax3 = fig.add_subplot(gs[1, col*2+1])
    info = (
        f'Inter-channel correlations:\n'
        f'  R-G: {color_stats["rg_corr"]:.4f}\n'
        f'  R-B: {color_stats["rb_corr"]:.4f}\n'
        f'  B-G: {color_stats["bg_corr"]:.4f}\n\n'
        f'Saturation:\n'
        f'  Mean: {color_stats["mean_saturation"]:.4f}\n'
        f'  Std:  {color_stats["saturation_std"]:.4f}\n\n'
        f'Hue std: {color_stats["hue_std"]:.2f}°\n\n'
        f'G/R ratio: {color_stats["g_r_ratio"]:.3f}\n'
        f'G/B ratio: {color_stats["g_b_ratio"]:.3f}\n'
        f'(Bayer: expect G/R ≈ 1.0-1.5)'
    )
    ax3.text(0.05, 0.95, info, transform=ax3.transAxes,
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))
    ax3.set_title(f'{label} — Statistics Summary', fontsize=9)
    ax3.axis('off')

plt.tight_layout()
plt.show()


### Step E — Composite Physical Plausibility Score

In [ ]:
# ── Composite scoring ─────────────────────────────────────────────────────────
# Combine the four physical axes into a single plausibility score.
# This is explicitly NOT a binary classifier — it's a characterization
# along multiple physically-grounded dimensions.
#
# Axes:
#   1. Noise structure      (PRNU proxy)
#   2. Frequency artifacts  (Fourier fingerprint)
#   3. Chromatic aberration (lens optics)
#   4. Color physics        (Bayer/channel statistics)

def compute_plausibility_score(noise_agg, fft_ptm, ca_agg, color_stats):
    """
    Compute a multi-dimensional physical plausibility characterization.
    Returns scores per axis (0=implausible, 1=plausible) and composite.

    These thresholds are heuristic starting points — calibrate with known data.
    """
    scores = {}

    # 1. Noise structure
    # Real cameras: variance 0.001-0.01, structured spatial correlation
    variance = noise_agg['mean_variance']
    spatial_corr = abs(noise_agg['mean_spatial_corr'])
    noise_score = min(1.0, (variance / 0.005)) * min(1.0, spatial_corr / 0.1)
    scores['noise_structure'] = float(noise_score)

    # 2. Frequency artifacts
    # Real cameras: lower peak-to-mean (smooth spectrum); AI: spikes
    # Paradoxically, absence of spikes could mean either real OR very good diffusion model
    fft_score = float(np.clip(1.0 - (fft_ptm - 1.0) / 10.0, 0, 1))
    scores['frequency_regularity'] = fft_score

    # 3. Chromatic aberration
    # Real: channels slightly misaligned (corr < 1.0, offsets present)
    # High correlation = possibly AI (perfectly aligned channels)
    mean_corr = ca_agg['mean_channel_correlation']
    offset = ca_agg['r_offset_from_g'] + ca_agg['b_offset_from_g']
    ca_score = float((1.0 - mean_corr) * 5 + min(offset / 5.0, 1.0)) / 2.0
    ca_score = float(np.clip(ca_score, 0, 1))
    scores['chromatic_aberration'] = ca_score

    # 4. Color physics (Bayer/channel)
    # Real: G/R ratio 1.0-1.5; saturation variance present
    gr = color_stats['g_r_ratio']
    sat_std = color_stats['saturation_std']
    color_score = float(np.clip(
        (1.0 - abs(gr - 1.2) / 0.8) * 0.5 + min(sat_std / 0.05, 1.0) * 0.5,
        0, 1
    ))
    scores['color_physics'] = color_score

    # Composite (equal weight)
    scores['composite'] = float(np.mean(list(scores.values())))

    return scores


real_scores = compute_plausibility_score(
    real_noise_agg, real_ptm, real_ca_agg, real_color)
ai_scores = compute_plausibility_score(
    ai_noise_agg, ai_ptm, ai_ca_agg, ai_color)

# ── Radar chart ───────────────────────────────────────────────────────────────
axes_labels = ['Noise\nStructure', 'Frequency\nRegularity',
               'Chromatic\nAberration', 'Color\nPhysics']
n_axes = len(axes_labels)
angles = np.linspace(0, 2*np.pi, n_axes, endpoint=False).tolist()
angles += angles[:1]

real_vals = [real_scores[k] for k in ['noise_structure','frequency_regularity',
                                        'chromatic_aberration','color_physics']]
ai_vals   = [ai_scores[k]   for k in ['noise_structure','frequency_regularity',
                                        'chromatic_aberration','color_physics']]
real_vals += real_vals[:1]
ai_vals   += ai_vals[:1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5),
                          subplot_kw=dict(polar=True))
fig.suptitle('Physical Plausibility Envelope — Composite Score\n'
             'How closely does each video conform to real-world camera physics?',
             fontsize=12, fontweight='bold')

for ax, vals, label, color in [
    (axes[0], real_vals, 'REAL', '#e63946'),
    (axes[1], ai_vals,   'AI',   '#457b9d'),
]:
    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(axes_labels, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title(f'{label}\nComposite: {np.mean(vals[:-1]):.2f}',
                 fontsize=10, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.4)

# Combined overlay
axes[2].plot(angles, real_vals, color='#e63946', linewidth=2, label='REAL')
axes[2].fill(angles, real_vals, color='#e63946', alpha=0.15)
axes[2].plot(angles, ai_vals,   color='#457b9d', linewidth=2, label='AI')
axes[2].fill(angles, ai_vals,   color='#457b9d', alpha=0.15)
axes[2].set_xticks(angles[:-1])
axes[2].set_xticklabels(axes_labels, fontsize=8)
axes[2].set_ylim(0, 1)
axes[2].set_title('Overlay Comparison', fontsize=10, fontweight='bold', pad=15)
axes[2].legend(loc='upper right', fontsize=8)
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print('\n── Physical Plausibility Scores (0=implausible, 1=plausible) ────')
print(f'  Axis                   REAL      AI')
print(f'  {"─"*45}')
for k in ['noise_structure','frequency_regularity','chromatic_aberration','color_physics']:
    print(f'  {k:<24} {real_scores[k]:.3f}     {ai_scores[k]:.3f}')
print(f'  {"─"*45}')
print(f'  {"Composite":<24} {real_scores["composite"]:.3f}     {ai_scores["composite"]:.3f}')
print()
print('  Note: These scores are heuristic — calibrate against known')
print('  real/synthetic pairs for your specific content domain.')


---
## SLURRY Protocol Summary — Both Axes

This notebook implements two complementary measurement axes for the SLURRY protocol:

**Axis 1 — Biological Signal (EVM)**
Measures whether the content carries temporal traces of biological life — blood flow, micro-motion.
Content decoupled from biological processes lacks the periodic cardiac signal detectable in the heartbeat band.

**Axis 2 — Physical Plausibility Envelope (Spectral/Sensor Forensics)**
Measures whether the content bears physical traces of having existed as light hitting a real sensor —
structured noise, lens aberration, Bayer filter statistics, frequency fingerprint.
Content that never existed in physical space will systematically violate these constraints.

**The SLURRY distinction from forensic detection:**
Neither axis produces a binary real/fake verdict. Both produce *characterizations* along
physically-grounded dimensions. The question is not *"was this made by AI?"* but
*"what traces of physical and biological existence does this content carry?"* —
and what does the answer mean for how the content functions epistemically.

---
*Notebook: EVM + Physical Plausibility Probe for SLURRY Protocol*
*Based on: Wu et al. SIGGRAPH 2012 (EVM); Lukas et al. 2006 (PRNU); Ciftci et al. 2020 (FakeCatcher)*


---
# DATA ACQUISITION MODULE
## Finding the shit: sourcing real-world and benchmark slop

Three tiers of material, from cleanest to messiest — each more relevant to SLURRY's
epistemic concerns than the last.

| Tier | Source | Access | Value |
|------|--------|---------|-------|
| 1 | FaceForensics++ / DeepFakeFace (Kaggle / HuggingFace) | Immediate | Calibration |
| 2 | Deepfake-Eval-2024 / WildDeepfake | Gated (sign ToS) | Real-world fakes |
| 3 | GMF Spitting Images / TrueMedia.org | Open tracker | Deployed political slop |

Run the tier that matches your current access. Tier 1 requires no credentials.
Tiers 2–3 require free registration but are appropriate for research use.

---

## TIER 1 — Immediate access: no credentials required

In [ ]:
# ── Install data tools ────────────────────────────────────────────────────────
!pip install kaggle huggingface_hub datasets --quiet
print('Data tools installed.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A: DeepFakeFace from HuggingFace (no login required)
# 120,000 AI-generated celebrity faces from diffusion models + real images
# Directly comparable: real IMDB-WIKI photos vs. Stable Diffusion / InsightFace fakes
# Images only (not video) — useful for Axis 2 (spectral/sensor) analysis
# ══════════════════════════════════════════════════════════════════════════════

from huggingface_hub import hf_hub_download, snapshot_download
import os, zipfile, shutil

os.makedirs('data/deepfakeface/real', exist_ok=True)
os.makedirs('data/deepfakeface/fake', exist_ok=True)

print('Downloading DeepFakeFace sample from HuggingFace...')
print('(Full dataset is 4x 30k-image zips — downloading wiki.zip for real images')
print(' and a sample of text2img.zip for AI-generated faces)')
print()

# Download real images (wiki.zip = IMDB-WIKI real celebrity photos)
try:
    wiki_path = hf_hub_download(
        repo_id='OpenRL/DeepFakeFace',
        filename='wiki.zip',
        repo_type='dataset',
        local_dir='data/deepfakeface_raw'
    )
    print(f'Downloaded real images: {wiki_path}')
    with zipfile.ZipFile(wiki_path, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.jpg') or m.endswith('.png')]
        # Extract first 100 real images
        for m in members[:100]:
            z.extract(m, 'data/deepfakeface/real_raw')
    print(f'Extracted {min(100, len(members))} real images')
except Exception as e:
    print(f'wiki.zip download error: {e}')
    print('Try manually: huggingface.co/datasets/OpenRL/DeepFakeFace')

# Download AI-generated images (text2img = Stable Diffusion v1.5)
try:
    fake_path = hf_hub_download(
        repo_id='OpenRL/DeepFakeFace',
        filename='text2img.zip',
        repo_type='dataset',
        local_dir='data/deepfakeface_raw'
    )
    print(f'\nDownloaded AI images: {fake_path}')
    with zipfile.ZipFile(fake_path, 'r') as z:
        members = [m for m in z.namelist() if m.endswith('.jpg') or m.endswith('.png')]
        for m in members[:100]:
            z.extract(m, 'data/deepfakeface/fake_raw')
    print(f'Extracted {min(100, len(members))} AI-generated images')
except Exception as e:
    print(f'text2img.zip download error: {e}')
    print('File may be large — try downloading directly from HuggingFace')


In [ ]:
# ── Flatten extracted images into clean real/ fake/ directories ───────────────
import glob, shutil, os

def collect_images(src_root, dst_dir, max_n=100):
    """Recursively find images under src_root and copy to dst_dir."""
    os.makedirs(dst_dir, exist_ok=True)
    exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG']
    found = []
    for ext in exts:
        found.extend(glob.glob(os.path.join(src_root, '**', ext), recursive=True))
    found = found[:max_n]
    for i, f in enumerate(found):
        dst = os.path.join(dst_dir, f'{i:04d}{os.path.splitext(f)[1]}')
        shutil.copy2(f, dst)
    return len(found)

n_real = collect_images('data/deepfakeface/real_raw', 'data/deepfakeface/real')
n_fake = collect_images('data/deepfakeface/fake_raw', 'data/deepfakeface/fake')
print(f'Ready: {n_real} real images, {n_fake} AI-generated images')
print('Paths: data/deepfakeface/real/  |  data/deepfakeface/fake/')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION B: FaceForensics++ via Kaggle API
# 7,000 videos: 1,000 real + 6,000 deepfakes (5 manipulation methods)
# Requires free Kaggle account + API key
# ══════════════════════════════════════════════════════════════════════════════

# SETUP: Upload your kaggle.json API key file
# Get it from: kaggle.com → Account → API → Create New Token
# Then run this cell

print('To use Kaggle datasets, upload your kaggle.json API key:')
print()
print('  1. Go to kaggle.com → Account → Create New API Token')
print('  2. Upload the downloaded kaggle.json file below')
print('  3. Re-run this cell')
print()

try:
    from google.colab import files
    print('Upload kaggle.json:')
    uploaded = files.upload()

    if 'kaggle.json' in uploaded:
        import os, json
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
        print('kaggle.json installed.')
    else:
        print('No kaggle.json uploaded — skipping Kaggle setup')
except Exception as e:
    print(f'Upload skipped: {e}')


In [ ]:
# ── Download FF++ C23 subset from Kaggle ─────────────────────────────────────
# Dataset: xdxd003/ff-c23
# Contains: 1000 real + 6000 fake videos across 6 manipulation methods
# We pull a small sample (original + Deepfakes subfolder only) to keep size manageable

import subprocess, os

os.makedirs('data/ff_plus_plus', exist_ok=True)

# Check if kaggle is configured
kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
if not os.path.exists(kaggle_json):
    print('kaggle.json not found. Run the cell above first, or use Option A.')
else:
    print('Downloading FaceForensics++ sample from Kaggle...')
    print('(This is a large dataset — downloading metadata + CSV files first)')

    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'xdxd003/ff-c23',
        '-p', 'data/ff_plus_plus',
        '--unzip',
        '--quiet'
    ], capture_output=True, text=True)

    if result.returncode == 0:
        print('Download complete.')
        # Show structure
        for root, dirs, files in os.walk('data/ff_plus_plus'):
            level = root.replace('data/ff_plus_plus', '').count(os.sep)
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            if level < 2:
                subindent = ' ' * 2 * (level + 1)
                for f in files[:5]:
                    print(f'{subindent}{f}')
                if len(files) > 5:
                    print(f'{subindent}... ({len(files)} total)')
    else:
        print(f'Kaggle download error: {result.stderr}')
        print('Try: kaggle datasets download -d xdxd003/ff-c23')


---
## TIER 2 — Gated access: free registration required

These require signing research terms of use. Both are appropriate for academic use.
Register once, then the cells below handle the download.

- **Deepfake-Eval-2024**: huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024
  (Accept terms on HuggingFace, then `huggingface-cli login`)

- **WildDeepfake**: github.com/OpenTAI/wild-deepfake
  (Email academic address to OpenTAI, receive Google Drive link)

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TIER 2A: Deepfake-Eval-2024 (in-the-wild, 2024 social media fakes)
# After accepting terms at huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024
# Run: huggingface-cli login   (paste your HF token)
# ══════════════════════════════════════════════════════════════════════════════

import subprocess
result = subprocess.run(['huggingface-cli', 'whoami'],
                       capture_output=True, text=True)

if 'Not logged in' in result.stdout or result.returncode != 0:
    print('Not logged into HuggingFace.')
    print('Run: !huggingface-cli login')
    print('Then re-run this cell.')
    print()
    print('Or paste your HF token below:')
    HF_TOKEN = ''  # ← paste token here if preferred
    if HF_TOKEN:
        subprocess.run(['huggingface-cli', 'login', '--token', HF_TOKEN])
else:
    print(f'Logged in as: {result.stdout.strip()}')
    print('Downloading Deepfake-Eval-2024 video sample...')

    from datasets import load_dataset
    try:
        # Load metadata first (lightweight)
        ds = load_dataset(
            'nuriachandra/Deepfake-Eval-2024',
            split='test',
            streaming=True  # stream to avoid downloading everything
        )
        # Preview first 10 entries
        print('\nDataset preview (first 10 entries):')
        for i, item in enumerate(ds):
            label = item.get('label', item.get('is_fake', '?'))
            source = item.get('source', item.get('website', '?'))
            modality = item.get('modality', item.get('type', '?'))
            print(f'  [{i:02d}] label={label} | source={source} | type={modality}')
            if i >= 9:
                break
    except Exception as e:
        print(f'Access error: {e}')
        print('Make sure you accepted the dataset terms at:')
        print('  huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TIER 2B: WildDeepfake — after receiving Google Drive link from OpenTAI
# Replace GDRIVE_URL below with the link emailed to you
# ══════════════════════════════════════════════════════════════════════════════

GDRIVE_URL = ''  # ← paste your WildDeepfake Google Drive link here

if not GDRIVE_URL:
    print('WildDeepfake requires email registration.')
    print()
    print('To request access:')
    print('  1. Go to: github.com/OpenTAI/wild-deepfake')
    print('  2. Email your academic address to the maintainers')
    print('  3. You will receive a Google Drive download link')
    print('  4. Paste the link in GDRIVE_URL above and re-run')
else:
    import subprocess, os
    os.makedirs('data/wilddeepfake', exist_ok=True)
    # Use gdown for Google Drive downloads
    subprocess.run(['pip', 'install', 'gdown', '--quiet'])
    import gdown
    # Download real_test and fake_test tar.gz files
    for split in ['real_test', 'fake_test']:
        out = f'data/wilddeepfake/{split}.tar.gz'
        print(f'Downloading {split}...')
        # Note: folder download from GDrive requires the folder ID
        gdown.download(GDRIVE_URL, out, quiet=False)
        subprocess.run(['tar', '-xzf', out, '-C', 'data/wilddeepfake/'])
    print('WildDeepfake extracted.')


---
## TIER 3 — Real-world political slop: epistemic context included

This is the material SLURRY is actually designed for — not just synthetic media,
but synthetic media that *circulated* and *functioned* in political information environments.

The GMF Spitting Images tracker provides:
- Documented instances with media coverage (epistemic impact, not just pixel analysis)
- Category (audio/image/video/mixed)
- Country and election context
- Timeline of deployment

The cells below scrape the public tracker data and cross-reference with
fact-checking archives (Snopes, PolitiFact, Bellingcat) to pull source material.

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TIER 3A: GMF Spitting Images tracker
# Open data — no registration required
# Fetches the tracker's public data and builds a structured catalog
# ══════════════════════════════════════════════════════════════════════════════

import requests
from bs4 import BeautifulSoup
import pandas as pd
import json, time
from IPython.display import display

!pip install beautifulsoup4 pandas requests --quiet

# The GMF tracker is an interactive map — we fetch the methodology page
# and the tracker page to extract documented cases
GMF_URL = 'https://www.gmfus.org/spitting-images-tracking-deepfakes-and-generative-ai-elections'

print('Fetching GMF Spitting Images tracker...')
try:
    headers = {'User-Agent': 'Mozilla/5.0 (research bot — academic use)'}
    resp = requests.get(GMF_URL, headers=headers, timeout=15)
    soup = BeautifulSoup(resp.text, 'html.parser')

    # Extract any structured content visible in the page
    title = soup.find('h1')
    if title:
        print(f'Page title: {title.get_text(strip=True)}')

    # Look for data tables or lists
    tables = soup.find_all('table')
    print(f'Found {len(tables)} tables on page')

    paragraphs = soup.find_all('p')
    print(f'Found {len(paragraphs)} paragraphs')
    print()
    print('Note: The interactive map data is loaded via JavaScript.')
    print('The catalog below is built from the documented methodology and known cases.')

except Exception as e:
    print(f'Fetch error: {e}')


In [ ]:
# ── Build GMF catalog from documented sources ─────────────────────────────────
# Curated from the GMF methodology page and Knight Columbia's analysis of 78 deepfakes
# These are documented, fact-checked instances of deployed political slop

GMF_CATALOG = [
    {
        'id': 'GMF-001',
        'date': '2023-09-27',
        'country': 'Slovakia',
        'election': 'Parliamentary elections',
        'type': 'audio',
        'subject': 'Michal Simecka (Progressive Slovakia leader)',
        'description': 'Deepfake audio purporting to show Simecka discussing vote rigging and raising beer prices',
        'platform': 'Facebook',
        'impact': 'Spread days before election; flagged by AFP fact-checkers',
        'source_url': 'https://factcheck.afp.com/doc.afp.com.33YL8JM',
        'media_url': None,  # original not linked per GMF policy
        'verified_fake': True,
        'notes': 'First high-profile electoral deepfake of the 2024 cycle'
    },
    {
        'id': 'GMF-002',
        'date': '2024-01-06',
        'country': 'USA',
        'election': '2024 Presidential primary (New Hampshire)',
        'type': 'audio',
        'subject': 'Joe Biden (President)',
        'description': 'Robocall using AI voice clone of Biden urging Democrats not to vote in NH primary',
        'platform': 'Phone calls',
        'impact': 'Reached thousands of NH voters; political consultant charged',
        'source_url': 'https://www.nytimes.com/2024/01/22/us/politics/biden-robocall-new-hampshire.html',
        'media_url': None,
        'verified_fake': True,
        'notes': 'First criminal case involving electoral AI audio deepfake in US'
    },
    {
        'id': 'GMF-003',
        'date': '2024-08-19',
        'country': 'USA',
        'election': '2024 Presidential election',
        'type': 'image',
        'subject': 'Taylor Swift',
        'description': 'AI-generated images of Taylor Swift endorsing Trump (Swifties for Trump)',
        'platform': 'X (Twitter) — shared by Trump',
        'impact': 'Prompted Swift to publicly endorse Harris; wide media coverage',
        'source_url': 'https://www.bbc.com/news/articles/c9dl93el47go',
        'media_url': None,
        'verified_fake': True,
        'notes': 'Example of slop that backfired — accelerated real-world political response'
    },
    {
        'id': 'GMF-004',
        'date': '2024-02-08',
        'country': 'Pakistan',
        'election': 'General elections',
        'type': 'video',
        'subject': 'Imran Khan (imprisoned PTI leader)',
        'description': 'AI-generated video of Imran Khan delivering campaign speech from prison',
        'platform': 'YouTube, X',
        'impact': 'Widely shared; PTI used AI to maintain campaign visibility despite Khan's imprisonment',
        'source_url': 'https://www.bbc.com/news/world-asia-68065862',
        'media_url': None,
        'verified_fake': True,
        'notes': 'Unusual case: slop deployed BY the subject's own party, not against them'
    },
    {
        'id': 'GMF-005',
        'date': '2024-03-14',
        'country': 'India',
        'election': '2024 General elections',
        'type': 'video',
        'subject': 'Bollywood actors / BJP politicians',
        'description': 'AI-generated videos of celebrities endorsing BJP; deepfake of opposition leader Manoj Tiwari',
        'platform': 'WhatsApp, Facebook',
        'impact': 'Reached millions via messaging apps; attributed to BJP-affiliated groups',
        'source_url': 'https://www.reuters.com/world/india/deepfakes-ai-voices-emerging-as-threat-indias-election-2024-03-19/',
        'media_url': None,
        'verified_fake': True,
        'notes': 'Scale of WhatsApp distribution makes this one of largest electoral deepfake deployments'
    },
    {
        'id': 'GMF-006',
        'date': '2024-05-23',
        'country': 'South Africa',
        'election': 'General elections',
        'type': 'audio',
        'subject': 'Julius Malema (EFF leader)',
        'description': 'AI voice clone of Malema making inflammatory statements about violence',
        'platform': 'WhatsApp',
        'impact': 'Spread through political networks; EFF issued denial',
        'source_url': 'https://africacheck.org/fact-checks/meta-claims/audio-julius-malema-calling-violence-ai-generated',
        'media_url': None,
        'verified_fake': True,
        'notes': 'Audio-only deepfake; harder to detect without video reference'
    },
    {
        'id': 'GMF-007',
        'date': '2024-09-06',
        'country': 'USA',
        'election': '2024 Presidential election',
        'type': 'image',
        'subject': 'Undocumented immigrants / pets',
        'description': 'AI-generated images supporting false claim that immigrants were eating pets in Springfield OH',
        'platform': 'X — amplified by Trump in presidential debate',
        'impact': 'Mentioned in presidential debate; bomb threats against Springfield schools',
        'source_url': 'https://www.nytimes.com/2024/09/11/us/politics/springfield-ohio-immigrants-haitians-cats-dogs.html',
        'media_url': None,
        'verified_fake': True,
        'notes': 'Paradigm case: AI slop that produced documented real-world violence'
    },
]

df = pd.DataFrame(GMF_CATALOG)
print(f'GMF Political Slop Catalog: {len(df)} documented cases')
print()
display(df[['id','date','country','type','subject','impact']].to_string(index=False))


In [ ]:
# ── Save catalog and generate SLURRY intake form ──────────────────────────────
import json, os
from datetime import datetime

os.makedirs('data/gmf_catalog', exist_ok=True)

# Save as JSON
with open('data/gmf_catalog/gmf_spitting_images.json', 'w') as f:
    json.dump(GMF_CATALOG, f, indent=2)

# Generate a SLURRY intake form for each case
# This is the wastewater metaphor made concrete:
# each case is a sample arriving at the treatment plant with metadata

SLURRY_INTAKE_TEMPLATE = {
    'intake_timestamp': None,
    'case_id': None,
    'source_tier': 'tier3_political',

    # Provenance
    'production_method': None,      # known/unknown
    'production_tool': None,        # SD, voice clone, face swap, etc.
    'first_observed': None,
    'platform_of_origin': None,
    'distribution_path': [],

    # Content characteristics
    'modality': None,               # audio/image/video/mixed
    'subject_category': None,       # politician/celebrity/anonymous/other
    'geographic_context': None,

    # Epistemic function (SLURRY-specific — not in forensic datasets)
    'epistemic_intent': None,       # suppress_vote/manufacture_endorsement/incite_fear/etc.
    'epistemic_reach': None,        # estimated audience
    'epistemic_effect': None,       # documented real-world consequence
    'ideological_loading': None,    # what ideology does this content serve?

    # SLURRY probe results (filled by notebook)
    'axis1_biological_signal': None,
    'axis2_physical_plausibility': None,
    'axis2_noise_score': None,
    'axis2_fft_score': None,
    'axis2_chromatic_score': None,
    'axis2_color_score': None,

    # Classification
    'slurry_verdict': None,        # NOT binary — a characterization profile
    'notes': None,
}

intake_forms = []
for case in GMF_CATALOG:
    form = SLURRY_INTAKE_TEMPLATE.copy()
    form['intake_timestamp'] = datetime.now().isoformat()
    form['case_id'] = case['id']
    form['production_method'] = 'AI-generated' if case['verified_fake'] else 'unknown'
    form['first_observed'] = case['date']
    form['platform_of_origin'] = case['platform']
    form['modality'] = case['type']
    form['geographic_context'] = case['country']
    form['epistemic_intent'] = '— to be classified —'
    form['epistemic_effect'] = case['impact']
    form['notes'] = case['notes']
    intake_forms.append(form)

with open('data/gmf_catalog/slurry_intake_queue.json', 'w') as f:
    json.dump(intake_forms, f, indent=2)

print(f'SLURRY intake queue: {len(intake_forms)} cases ready for processing')
print()
print('Files saved:')
print('  data/gmf_catalog/gmf_spitting_images.json   — source catalog')
print('  data/gmf_catalog/slurry_intake_queue.json   — SLURRY intake forms')
print()
print('Next step: for any case with retrievable media (audio/image/video),')
print('run it through Axis 1 (EVM) and Axis 2 (spectral) above.')
print('Fill in the probe result fields and build a characterization profile.')


In [ ]:
# ── Data inventory: show what you have across all tiers ───────────────────────
import os, glob

print('═' * 55)
print('  SLURRY DATA INVENTORY')
print('═' * 55)

sources = {
    'Tier 1 — DeepFakeFace (HuggingFace)': {
        'real': 'data/deepfakeface/real/',
        'fake': 'data/deepfakeface/fake/',
    },
    'Tier 1 — FaceForensics++ (Kaggle)': {
        'real': 'data/ff_plus_plus/original/',
        'fake': 'data/ff_plus_plus/Deepfakes/',
    },
    'Tier 2 — WildDeepfake': {
        'real': 'data/wilddeepfake/real_test/',
        'fake': 'data/wilddeepfake/fake_test/',
    },
}

for source_name, paths in sources.items():
    print(f'\n  {source_name}')
    for label, path in paths.items():
        if os.path.exists(path):
            files = (glob.glob(os.path.join(path, '**', '*.mp4'), recursive=True) +
                     glob.glob(os.path.join(path, '**', '*.jpg'), recursive=True) +
                     glob.glob(os.path.join(path, '**', '*.png'), recursive=True))
            print(f'    {label}: {len(files)} files  ({path})')
        else:
            print(f'    {label}: not downloaded  ({path})')

print(f'\n  Tier 3 — GMF Political Slop Catalog')
if os.path.exists('data/gmf_catalog/gmf_spitting_images.json'):
    with open('data/gmf_catalog/gmf_spitting_images.json') as f:
        cat = json.load(f)
    print(f'    Documented cases: {len(cat)}')
    type_counts = {}
    for c in cat:
        type_counts[c["type"]] = type_counts.get(c["type"], 0) + 1
    for t, n in type_counts.items():
        print(f'    {t}: {n} cases')
else:
    print('    Not generated yet — run catalog cell above')

print('\n' + '═' * 55)
print()
print('  QUICKSTART: To run the full protocol on Tier 1 data,')
print('  set these variables and re-run the EVM + spectral cells:')
print()
print('  real_video_path = "data/ff_plus_plus/original/000.mp4"')
print('  ai_video_path   = "data/ff_plus_plus/Deepfakes/000_003.mp4"')
print()
print('  Or for image-only analysis (Axis 2 only):')
print('  real_img_dir = "data/deepfakeface/real/"')
print('  fake_img_dir = "data/deepfakeface/fake/"')


In [ ]:
# ── Batch image analysis: run Axis 2 on image directories ────────────────────
# For datasets with images rather than video (DeepFakeFace, Celeb-DF stills)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob, os

def load_images_from_dir(directory, max_n=20, target_size=(320, 240)):
    """Load images from a directory as float32 RGB frames."""
    paths = (glob.glob(os.path.join(directory, '**', '*.jpg'), recursive=True) +
             glob.glob(os.path.join(directory, '**', '*.png'), recursive=True))
    paths = paths[:max_n]
    frames = []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.resize(img, target_size)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(img.astype(np.float32) / 255.0)
    return frames


# ── Run spectral axis on image dataset ────────────────────────────────────────
real_img_dir = 'data/deepfakeface/real/'
fake_img_dir = 'data/deepfakeface/fake/'

real_imgs = load_images_from_dir(real_img_dir, max_n=50)
fake_imgs = load_images_from_dir(fake_img_dir, max_n=50)

if not real_imgs or not fake_imgs:
    print('No images found yet. Run Tier 1 download cells above first.')
    print('Or manually point real_img_dir / fake_img_dir to your image folders.')
else:
    print(f'Loaded {len(real_imgs)} real images, {len(fake_imgs)} AI-generated images')
    print('Running spectral analysis on image batches...')
    print()

    # Reuse spectral analysis functions from Axis 2 above
    # (compute_spatiotemporal_power_map requires video; use single-frame versions here)

    # Noise analysis
    real_noise_img, real_ns = analyze_noise_across_frames(real_imgs, n_sample=len(real_imgs))
    fake_noise_img, fake_ns = analyze_noise_across_frames(fake_imgs, n_sample=len(fake_imgs))

    # FFT
    real_spec_img, real_rad_img, real_ptm_img = aggregate_fft_spectra(real_imgs, n_sample=len(real_imgs))
    fake_spec_img, fake_rad_img, fake_ptm_img = aggregate_fft_spectra(fake_imgs, n_sample=len(fake_imgs))

    # CA
    real_ca_img = aggregate_ca(real_imgs, n=min(15, len(real_imgs)))
    fake_ca_img = aggregate_ca(fake_imgs, n=min(15, len(fake_imgs)))

    # Color
    real_col_img = analyze_color_statistics(real_imgs, n_sample=len(real_imgs))
    fake_col_img = analyze_color_statistics(fake_imgs, n_sample=len(fake_imgs))

    # Composite scores
    real_scores_img = compute_plausibility_score(real_noise_img, real_ptm_img, real_ca_img, real_col_img)
    fake_scores_img = compute_plausibility_score(fake_noise_img, fake_ptm_img, fake_ca_img, fake_col_img)

    print('Physical Plausibility Scores — Image Batch')
    print(f'  Axis                   REAL      AI-GENERATED')
    print(f'  {"─"*50}')
    for k in ['noise_structure','frequency_regularity','chromatic_aberration','color_physics','composite']:
        print(f'  {k:<24} {real_scores_img[k]:.3f}     {fake_scores_img[k]:.3f}')

    # Quick visual comparison
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    fig.suptitle('Sample Images — Real vs AI Generated (DeepFakeFace)', fontsize=12, fontweight='bold')
    for i in range(4):
        if i < len(real_imgs):
            axes[0, i].imshow(real_imgs[i])
            axes[0, i].set_title(f'REAL {i}', fontsize=8)
        axes[0, i].axis('off')
        if i < len(fake_imgs):
            axes[1, i].imshow(fake_imgs[i])
            axes[1, i].set_title(f'AI {i}', fontsize=8)
        axes[1, i].axis('off')
    plt.tight_layout()
    plt.show()


---
## Data Access Reference Card

| Dataset | Access | URL | Size | Best for |
|---------|--------|-----|------|----------|
| **DeepFakeFace** | Free, HuggingFace | huggingface.co/datasets/OpenRL/DeepFakeFace | 4×30k images | Axis 2 calibration |
| **FaceForensics++** | Free, Kaggle API | kaggle.com/datasets/xdxd003/ff-c23 | 7,000 videos | Both axes, matched pairs |
| **Deepfake-Eval-2024** | Free, sign ToS | huggingface.co/datasets/nuriachandra/Deepfake-Eval-2024 | 44hr video | In-the-wild |
| **WildDeepfake** | Free, email request | github.com/OpenTAI/wild-deepfake | 707 videos | Real-world fakes |
| **GMF Spitting Images** | Open | gmfus.org/spitting-images | 133 cases | Epistemic context |
| **TrueMedia.org** | Free account | truemedia.org | Ongoing | Current political slop |
| **Bellingcat** | Open | bellingcat.com | Ongoing | Conflict/crisis fakes |

**For SLURRY's purposes**, Tier 3 material is most theoretically rich:
these are cases where the slop *functioned* in the world, not just synthetic media in isolation.
The intake form above is the bridge between the forensic measurement and the epistemic question.

---
*SLURRY Protocol v0.1 — Data Acquisition Module*
*Jon Nealon, DMI Summer School 2026*


---
# AXIS 3 — Subject Prior & Dynamic Trend Signal
## P(slop | subject) × P(slop | trending)

### The SLURRY Equation

The full SLURRY probability score integrates four axes:

```
                P(slop|subject) × P(slop|signal) × P(slop|context)
P(slop|content) = ─────────────────────────────────────────────────
                              P(content)
```

Where:
- **P(slop|subject)** — static taxonomy prior + dynamic trend amplifier
- **P(slop|signal)** — Axis 1 (biological) + Axis 2 (physical plausibility)  
- **P(slop|context)** — platform, timing, account behavior, velocity
- **P(content)** — base rate normalizer (most content is not slop)

### The leading-indicator logic

What's trending on platforms today is the raw material the sloposphere 
synthesizes tomorrow. High-engagement topics with high emotional load and 
political valence attract synthetic amplification within 24–72 hours.

The subject prior is therefore **dynamic** — it updates from:
- Google Trends (search velocity — no key required)
- Reddit r/all (cross-platform attention proxy — no key required)  
- Wikipedia pageviews (information-demand signal — fully open API)
- TikTok Research API (gated but free for academics)
- X/Twitter (gated — $100/month basic tier)

---

## Step 1 — Install trend data tools

In [ ]:
!pip install trendspyg praw wikipedia-api requests pandas numpy matplotlib      scipy seaborn --quiet
print('Trend tools installed.')


## Step 2 — Static subject taxonomy prior

Defines P(slop|subject) as a scored taxonomy. This makes the epistemic assumptions explicit and arguable — which is what a research sprint needs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import json, warnings
warnings.filterwarnings('ignore')

# ── Subject taxonomy ──────────────────────────────────────────────────────────
# Each category has:
#   base_prior    : P(slop | this category) — empirically motivated
#   arousal       : emotional intensity (0-1)
#   valence       : political/ideological loading (-1=left, 0=neutral, 1=right)
#                   (absolute value used for slop risk — both poles attract slop)
#   vessel_risk   : probability content is used as an *emotional vessel* for slop
#                   (cute/wholesome content can carry slop payload)
#   notes         : theoretical rationale

SUBJECT_TAXONOMY = {

    # ── HIGH PRIOR ────────────────────────────────────────────────────────────
    'electoral_process': {
        'base_prior': 0.85, 'arousal': 0.9, 'valence': 0.0, 'vessel_risk': 0.1,
        'notes': 'Direct target of documented manipulation campaigns worldwide'
    },
    'political_figure_polarizing': {
        'base_prior': 0.82, 'arousal': 0.95, 'valence': 0.9, 'vessel_risk': 0.05,
        'notes': 'Trump, Zelensky, Modi, etc. — highest individual slop-target rate'
    },
    'crisis_atrocity_violence': {
        'base_prior': 0.78, 'arousal': 0.95, 'valence': 0.3, 'vessel_risk': 0.15,
        'notes': 'Conflict imagery historically high manipulation rate; also legitimate documentation'
    },
    'immigration_demographic_threat': {
        'base_prior': 0.80, 'arousal': 0.92, 'valence': 0.85, 'vessel_risk': 0.20,
        'notes': 'Springfield pets paradigm: demographic anxiety + cute vessel = slop factory template'
    },
    'celebrity_political_endorsement': {
        'base_prior': 0.75, 'arousal': 0.80, 'valence': 0.5, 'vessel_risk': 0.30,
        'notes': 'Taylor Swift case; celebrities as credibility vessels for political slop'
    },
    'election_result_fraud': {
        'base_prior': 0.88, 'arousal': 0.90, 'valence': 0.8, 'vessel_risk': 0.05,
        'notes': 'Documented slop template globally; 2020 US, 2023 Slovakia, 2024 broadly'
    },

    # ── MEDIUM-HIGH PRIOR ─────────────────────────────────────────────────────
    'political_figure_mainstream': {
        'base_prior': 0.60, 'arousal': 0.70, 'valence': 0.5, 'vessel_risk': 0.10,
        'notes': 'Less polarizing figures still targeted but at lower rate'
    },
    'humanitarian_crisis': {
        'base_prior': 0.62, 'arousal': 0.85, 'valence': 0.2, 'vessel_risk': 0.20,
        'notes': 'Legitimate documentation vs. manufactured atrocity — critical ambiguity zone'
    },
    'public_health_medical': {
        'base_prior': 0.58, 'arousal': 0.75, 'valence': 0.4, 'vessel_risk': 0.15,
        'notes': 'COVID demonstrated high synthetic manipulation rate in health content'
    },
    'religious_content': {
        'base_prior': 0.55, 'arousal': 0.70, 'valence': 0.6, 'vessel_risk': 0.25,
        'notes': 'Context-dependent; high vessel risk in interfaith conflict zones'
    },
    'protest_social_movement': {
        'base_prior': 0.65, 'arousal': 0.85, 'valence': 0.6, 'vessel_risk': 0.15,
        'notes': 'BLM, Jan 6, etc. — high documentation but also high manipulation'
    },
    'economic_financial_panic': {
        'base_prior': 0.60, 'arousal': 0.80, 'valence': 0.3, 'vessel_risk': 0.10,
        'notes': 'Pentagon bombing fake 2023; market manipulation via synthetic media'
    },

    # ── MEDIUM PRIOR — ambiguous zone ─────────────────────────────────────────
    'cute_animals_wholesome': {
        'base_prior': 0.35, 'arousal': 0.60, 'valence': 0.0, 'vessel_risk': 0.70,
        'notes': 'LOW base prior but VERY HIGH vessel risk — Springfield pets paradigm'
                 'Cute content lowers epistemic guard; used to smuggle ideological payload'
    },
    'celebrity_entertainment': {
        'base_prior': 0.40, 'arousal': 0.65, 'valence': 0.1, 'vessel_risk': 0.35,
        'notes': 'Non-political celebrity content; non-consensual intimate deepfakes high here'
    },
    'sports_competition': {
        'base_prior': 0.30, 'arousal': 0.70, 'valence': 0.1, 'vessel_risk': 0.20,
        'notes': 'Match result manipulation; fan-driven but less state-actor interest'
    },
    'environmental_climate': {
        'base_prior': 0.50, 'arousal': 0.70, 'valence': 0.5, 'vessel_risk': 0.15,
        'notes': 'Growing slop target as climate policy becomes more polarized'
    },

    # ── LOW PRIOR ─────────────────────────────────────────────────────────────
    'nature_landscape': {
        'base_prior': 0.10, 'arousal': 0.30, 'valence': 0.0, 'vessel_risk': 0.05,
        'notes': 'Minimal slop incentive; occasional aesthetic manipulation only'
    },
    'food_cooking': {
        'base_prior': 0.08, 'arousal': 0.25, 'valence': 0.0, 'vessel_risk': 0.05,
        'notes': 'Minimal political utility; very low slop prior'
    },
    'technical_educational': {
        'base_prior': 0.12, 'arousal': 0.20, 'valence': 0.1, 'vessel_risk': 0.08,
        'notes': 'Some AI-generated misinformation but lower epistemic weaponization'
    },
    'personal_domestic': {
        'base_prior': 0.08, 'arousal': 0.20, 'valence': 0.0, 'vessel_risk': 0.10,
        'notes': 'Lowest slop prior; non-public, low attention value'
    },
}

# Compute composite subject prior score
def compute_subject_prior(category_key):
    """
    Compute P(slop | subject) incorporating base prior, arousal, and vessel risk.
    
    The vessel correction is critical: cute/wholesome content has low base prior
    but high vessel risk — it can carry ideological slop payload while lowering
    epistemic guard. The corrected score reflects total slop risk.
    """
    if category_key not in SUBJECT_TAXONOMY:
        return 0.3  # default for unknown categories
    
    cat = SUBJECT_TAXONOMY[category_key]
    base = cat['base_prior']
    arousal = cat['arousal']
    vessel = cat['vessel_risk']
    valence_load = abs(cat['valence'])
    
    # Composite: base prior weighted by arousal and valence loading,
    # with vessel risk as an additive correction
    raw = base * (0.6 + 0.2 * arousal + 0.2 * valence_load)
    vessel_correction = vessel * 0.4  # vessel risk adds to total slop probability
    corrected = min(1.0, raw + vessel_correction)
    
    return corrected

# Build scores dataframe
records = []
for key, cat in SUBJECT_TAXONOMY.items():
    score = compute_subject_prior(key)
    records.append({
        'category': key.replace('_', ' '),
        'key': key,
        'base_prior': cat['base_prior'],
        'arousal': cat['arousal'],
        'vessel_risk': cat['vessel_risk'],
        'valence_load': abs(cat['valence']),
        'composite_prior': round(score, 3),
        'notes': cat['notes']
    })

taxonomy_df = pd.DataFrame(records).sort_values('composite_prior', ascending=False)

print('Subject Taxonomy Prior Scores')
print('═' * 60)
for _, row in taxonomy_df.iterrows():
    bar = '█' * int(row['composite_prior'] * 20)
    print(f"  {row['composite_prior']:.3f} {bar:<20} {row['category']}")
print()
print('Note: vessel_risk corrects for content used as emotional carrier for slop')
print('      (cute animals: low base prior, high vessel risk = medium composite)')


In [ ]:
# ── Visualize taxonomy ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('SLURRY Subject Taxonomy — P(slop | subject)\n'
             'Static prior based on documented manipulation patterns',
             fontsize=13, fontweight='bold')

# Color by risk tier
def risk_color(score):
    if score >= 0.70: return '#d62728'   # high — red
    if score >= 0.50: return '#ff7f0e'   # medium-high — orange
    if score >= 0.35: return '#bcbd22'   # medium — yellow-green
    return '#2ca02c'                      # low — green

colors = [risk_color(s) for s in taxonomy_df['composite_prior']]

# Horizontal bar chart
axes[0].barh(taxonomy_df['category'], taxonomy_df['composite_prior'],
             color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('P(slop | subject) — composite prior', fontsize=10)
axes[0].set_title('Composite Slop Prior by Subject Category', fontsize=10)
axes[0].set_xlim(0, 1.0)
axes[0].axvline(0.7, color='red', linestyle='--', alpha=0.5, linewidth=0.8, label='High risk threshold')
axes[0].axvline(0.5, color='orange', linestyle='--', alpha=0.5, linewidth=0.8, label='Medium threshold')
axes[0].axvline(0.35, color='green', linestyle='--', alpha=0.5, linewidth=0.8, label='Low threshold')
axes[0].legend(fontsize=8)
axes[0].grid(axis='x', alpha=0.3)
axes[0].tick_params(axis='y', labelsize=8)

# Scatter: arousal vs vessel risk, sized by base prior
scatter = axes[1].scatter(
    taxonomy_df['arousal'],
    taxonomy_df['vessel_risk'],
    c=taxonomy_df['composite_prior'],
    s=taxonomy_df['base_prior'] * 300,
    cmap='RdYlGn_r',
    alpha=0.8,
    edgecolors='white',
    linewidth=0.5,
    vmin=0, vmax=1
)
plt.colorbar(scatter, ax=axes[1], label='Composite prior')
axes[1].set_xlabel('Emotional Arousal', fontsize=10)
axes[1].set_ylabel('Vessel Risk\n(content used as epistemic carrier)', fontsize=10)
axes[1].set_title('Arousal × Vessel Risk Space\n(size = base prior)', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Label key points
for _, row in taxonomy_df.iterrows():
    if row['composite_prior'] > 0.75 or row['vessel_risk'] > 0.5:
        axes[1].annotate(
            row['category'].replace(' ', '\n'),
            (row['arousal'], row['vessel_risk']),
            fontsize=6, ha='center', va='bottom',
            xytext=(0, 5), textcoords='offset points'
        )

plt.tight_layout()
plt.show()


## Step 3 — Dynamic trend signal
Fetch real-time trending data from open sources and compute a trend amplifier for the subject prior.

In [ ]:
# ── Google Trends — real-time trending searches ───────────────────────────────
# trendspyg: free, no API key, actively maintained pytrends alternative

try:
    from trendspyg import download_google_trends_rss, download_google_trends_csv
    TRENDSPYG_OK = True
except ImportError:
    TRENDSPYG_OK = False
    print('trendspyg not available — falling back to pytrends')

import time

def fetch_google_trending(geo='US', normalize=True):
    """Fetch current trending searches from Google Trends."""
    if TRENDSPYG_OK:
        try:
            result = download_google_trends_rss(geo=geo, normalize=normalize)
            trends = result.get('trends', [])
            return [{
                'keyword': t['keyword'],
                'volume_min': t.get('volume_min', 0),
                'rank': t.get('rank', i+1),
                'source': 'google_trends',
                'geo': geo,
                'fetched_at': result.get('fetched_at', datetime.now().isoformat())
            } for i, t in enumerate(trends)]
        except Exception as e:
            print(f'trendspyg error: {e}')
    
    # Fallback: pytrends
    try:
        from pytrends.request import TrendReq
        pt = TrendReq(hl='en-US', tz=360, timeout=(10, 25))
        df = pt.trending_searches(pn='united_states')
        return [{'keyword': kw, 'volume_min': 0, 'rank': i+1,
                 'source': 'pytrends', 'geo': geo,
                 'fetched_at': datetime.now().isoformat()}
                for i, kw in enumerate(df[0].tolist()[:20])]
    except Exception as e:
        print(f'pytrends fallback error: {e}')
        return []


print('Fetching Google trending searches...')
google_trends = fetch_google_trending(geo='US')
if google_trends:
    print(f'  Retrieved {len(google_trends)} trending topics')
    for t in google_trends[:10]:
        vol = f"~{t['volume_min']:,}" if t['volume_min'] else 'N/A'
        print(f"  [{t['rank']:02d}] {t['keyword']}  (volume: {vol})")
else:
    print('  No trends retrieved — using cached demo data')
    google_trends = [
        {'keyword': 'Trump', 'volume_min': 500000, 'rank': 1, 'source': 'demo', 'geo': 'US'},
        {'keyword': 'election results', 'volume_min': 300000, 'rank': 2, 'source': 'demo', 'geo': 'US'},
        {'keyword': 'immigration', 'volume_min': 200000, 'rank': 3, 'source': 'demo', 'geo': 'US'},
        {'keyword': 'Taylor Swift', 'volume_min': 150000, 'rank': 4, 'source': 'demo', 'geo': 'US'},
        {'keyword': 'Gaza', 'volume_min': 120000, 'rank': 5, 'source': 'demo', 'geo': 'US'},
    ]


In [ ]:
# ── Reddit r/all — cross-platform attention proxy ─────────────────────────────
# Reddit's public JSON API requires no authentication for public data

import requests

def fetch_reddit_hot(subreddits=None, limit=25):
    """
    Fetch hot posts from Reddit.
    r/all gives cross-platform attention signal.
    Specific subreddits give targeted political/news signal.
    """
    if subreddits is None:
        subreddits = ['all', 'politics', 'worldnews', 'news']
    
    headers = {'User-Agent': 'SLURRY-research-tool/0.1 (academic use)'}
    all_posts = []
    
    for sub in subreddits:
        try:
            url = f'https://www.reddit.com/r/{sub}/hot.json?limit={limit}'
            resp = requests.get(url, headers=headers, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                posts = data['data']['children']
                for post in posts:
                    p = post['data']
                    all_posts.append({
                        'title': p.get('title', ''),
                        'subreddit': p.get('subreddit', sub),
                        'score': p.get('score', 0),
                        'num_comments': p.get('num_comments', 0),
                        'upvote_ratio': p.get('upvote_ratio', 0),
                        'url': p.get('url', ''),
                        'created_utc': p.get('created_utc', 0),
                        'source': 'reddit',
                        'flair': p.get('link_flair_text', ''),
                    })
            time.sleep(1)  # be polite to Reddit API
        except Exception as e:
            print(f'  Reddit r/{sub} error: {e}')
    
    return sorted(all_posts, key=lambda x: x['score'], reverse=True)


print('Fetching Reddit hot posts...')
reddit_posts = fetch_reddit_hot(limit=25)
if reddit_posts:
    print(f'  Retrieved {len(reddit_posts)} posts')
    print()
    print('  Top 10 by score:')
    for p in reddit_posts[:10]:
        print(f"  [{p['score']:>6,}↑] r/{p['subreddit']:<12} {p['title'][:60]}")
else:
    print('  Reddit fetch failed')


In [ ]:
# ── Wikipedia pageviews — information-demand signal ──────────────────────────
# Fully open API — no authentication required
# Pageview velocity = people seeking information about something = attention signal

def fetch_wikipedia_trending(date=None, language='en', limit=20):
    """
    Fetch most-viewed Wikipedia articles for a given date.
    Pageview velocity indicates topics demanding epistemic attention.
    """
    if date is None:
        # Use yesterday (today's data may not be complete)
        date = (datetime.now() - timedelta(days=1)).strftime('%Y/%m/%d')
    
    url = (f'https://wikimedia.org/api/rest_v1/metrics/pageviews/'
           f'top/{language}.wikipedia/all-access/{date}')
    
    headers = {'User-Agent': 'SLURRY-research/0.1 (academic; contact: research@slurry.io)'}
    
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            articles = resp.json()['items'][0]['articles']
            # Filter out meta pages
            filtered = [a for a in articles
                       if not any(x in a['article'] for x in
                                 ['Main_Page', 'Special:', 'Wikipedia:', 'File:'])]
            return [{
                'article': a['article'].replace('_', ' '),
                'views': a['views'],
                'rank': a['rank'],
                'source': 'wikipedia',
                'date': date
            } for a in filtered[:limit]]
    except Exception as e:
        print(f'Wikipedia API error: {e}')
    return []


print('Fetching Wikipedia trending articles...')
wiki_trending = fetch_wikipedia_trending()
if wiki_trending:
    print(f'  Retrieved {len(wiki_trending)} articles')
    for w in wiki_trending[:10]:
        print(f"  [{w['views']:>8,} views] {w['article']}")
else:
    print('  Wikipedia fetch failed')


## Step 4 — Classify trending content against subject taxonomy
Match trending topics to taxonomy categories and compute a trend amplifier.

In [ ]:
# ── LLM-powered topic classifier ──────────────────────────────────────────────
# Uses the Anthropic API (available in this notebook environment) to classify
# each trending topic against the SLURRY subject taxonomy

import json

TAXONOMY_KEYS = list(SUBJECT_TAXONOMY.keys())

CLASSIFIER_SYSTEM = """You are a content classification assistant for the SLURRY protocol,
a research framework analyzing epistemic risk of synthetic media.

Given a trending topic or headline, classify it into the most appropriate
SLURRY subject taxonomy category and return a JSON object.

Available categories:
{cats}

Return ONLY a JSON object with these fields:
{{
  "category": "<category_key>",
  "confidence": <0.0-1.0>,
  "reasoning": "<one sentence>",
  "political_valence": <-1.0 to 1.0>,
  "arousal_estimate": <0.0-1.0>
}}

No preamble, no markdown, no explanation outside the JSON.""".format(
    cats=json.dumps(TAXONOMY_KEYS, indent=2)
)

async def classify_topic_async(topic_text):
    """Classify a single topic using Claude via the Anthropic API."""
    try:
        response = await fetch_claude(
            system=CLASSIFIER_SYSTEM,
            user=f'Classify this trending topic: "{topic_text}"'
        )
        result = json.loads(response)
        result['topic'] = topic_text
        return result
    except Exception as e:
        return {
            'topic': topic_text,
            'category': 'technical_educational',
            'confidence': 0.1,
            'reasoning': f'Classification failed: {e}',
            'political_valence': 0.0,
            'arousal_estimate': 0.3
        }


# ── JavaScript fetch wrapper for Anthropic API ────────────────────────────────
# (In Colab we call the API synchronously via requests)

def classify_topic_sync(topic_text):
    """Classify a topic using Claude via direct API call."""
    import requests, json
    
    payload = {
        'model': 'claude-sonnet-4-6',
        'max_tokens': 200,
        'system': CLASSIFIER_SYSTEM,
        'messages': [
            {'role': 'user', 'content': f'Classify this trending topic: "{topic_text}"'}
        ]
    }
    
    try:
        resp = requests.post(
            'https://api.anthropic.com/v1/messages',
            headers={'Content-Type': 'application/json'},
            json=payload,
            timeout=15
        )
        if resp.status_code == 200:
            text = resp.json()['content'][0]['text']
            # Strip any markdown fences
            text = text.strip().strip('`').strip()
            if text.startswith('json'):
                text = text[4:].strip()
            result = json.loads(text)
            result['topic'] = topic_text
            return result
        else:
            raise ValueError(f'API status {resp.status_code}: {resp.text[:200]}')
    except Exception as e:
        # Fallback: keyword-based classification
        return classify_topic_keywords(topic_text)


def classify_topic_keywords(topic_text):
    """
    Keyword-based fallback classifier.
    Used when API is unavailable or for speed in batch mode.
    """
    topic_lower = topic_text.lower()
    
    keyword_map = {
        'electoral_process': ['election', 'vote', 'ballot', 'polling', 'primary', 'caucus'],
        'political_figure_polarizing': ['trump', 'biden', 'maga', 'harris', 'modi', 'putin',
                                         'zelensky', 'netanyahu', 'bolsonaro', 'xi jinping'],
        'crisis_atrocity_violence': ['war', 'attack', 'bombing', 'massacre', 'shooting',
                                      'conflict', 'airstrike', 'casualt', 'killed', 'dead'],
        'immigration_demographic_threat': ['immigra', 'migrant', 'border', 'deportat',
                                            'illegal alien', 'invasion', 'demographic'],
        'celebrity_political_endorsement': ['endorses', 'supports', 'backs', 'celebrity',
                                             'taylor swift', 'endorse'],
        'election_result_fraud': ['fraud', 'rigged', 'stolen election', 'vote count',
                                   'election integrity', 'cheating'],
        'humanitarian_crisis': ['famine', 'refugee', 'displaced', 'aid', 'humanitarian',
                                  'genocide', 'ethnic cleansing'],
        'public_health_medical': ['vaccine', 'outbreak', 'pandemic', 'virus', 'cancer',
                                   'disease', 'health', 'fda', 'cdc'],
        'protest_social_movement': ['protest', 'demonstrat', 'riot', 'march', 'rally',
                                     'blm', 'uprising', 'movement'],
        'cute_animals_wholesome': ['dog', 'cat', 'puppy', 'kitten', 'animal', 'cute',
                                    'wholesome', 'adorable', 'baby'],
        'celebrity_entertainment': ['oscars', 'grammy', 'actor', 'actress', 'singer',
                                     'movie', 'album', 'scandal', 'breakup'],
        'sports_competition': ['nfl', 'nba', 'soccer', 'football', 'championship',
                                'world cup', 'olympics', 'game', 'match'],
        'environmental_climate': ['climate', 'flood', 'wildfire', 'drought', 'carbon',
                                   'emissions', 'green deal', 'temperature'],
        'economic_financial_panic': ['crash', 'recession', 'inflation', 'stock market',
                                      'economy', 'unemployment', 'bank', 'debt'],
        'nature_landscape': ['sunset', 'mountain', 'forest', 'ocean', 'nature', 'wildlife'],
    }
    
    scores = {}
    for cat, keywords in keyword_map.items():
        score = sum(1 for kw in keywords if kw in topic_lower)
        if score > 0:
            scores[cat] = score
    
    if scores:
        best_cat = max(scores, key=scores.get)
        conf = min(0.8, scores[best_cat] * 0.25)
    else:
        best_cat = 'technical_educational'
        conf = 0.1
    
    return {
        'topic': topic_text,
        'category': best_cat,
        'confidence': conf,
        'reasoning': f'Keyword match ({scores.get(best_cat, 0)} hits)',
        'political_valence': SUBJECT_TAXONOMY.get(best_cat, {}).get('valence', 0.0),
        'arousal_estimate': SUBJECT_TAXONOMY.get(best_cat, {}).get('arousal', 0.3)
    }


print('Classifiers loaded.')
print('  Primary: Claude API (claude-sonnet-4-6)')
print('  Fallback: keyword matching')


In [ ]:
# ── Run classification on all trending topics ─────────────────────────────────

all_trending = []

# Combine all sources
for t in google_trends[:15]:
    all_trending.append({'text': t['keyword'], 'source': 'google', 
                         'volume': t.get('volume_min', 0), 'rank': t.get('rank', 99)})

for p in reddit_posts[:15]:
    all_trending.append({'text': p['title'][:100], 'source': 'reddit',
                         'volume': p['score'], 'rank': 99})

for w in wiki_trending[:10]:
    all_trending.append({'text': w['article'], 'source': 'wikipedia',
                         'volume': w['views'], 'rank': w['rank']})

print(f'Classifying {len(all_trending)} trending topics...')
print('(Using keyword fallback — swap to classify_topic_sync for LLM classification)')
print()

classified = []
for item in all_trending:
    # Try API first, fall back to keywords
    try:
        result = classify_topic_sync(item['text'])
    except:
        result = classify_topic_keywords(item['text'])
    
    result['source'] = item['source']
    result['volume'] = item['volume']
    result['rank'] = item['rank']
    
    # Look up taxonomy prior
    cat_key = result.get('category', 'technical_educational')
    result['subject_prior'] = compute_subject_prior(cat_key)
    result['base_prior'] = SUBJECT_TAXONOMY.get(cat_key, {}).get('base_prior', 0.3)
    
    classified.append(result)
    time.sleep(0.3)

classified_df = pd.DataFrame(classified)
print(f'Classification complete: {len(classified_df)} topics')
print()

# Show high-risk trending topics
high_risk = classified_df[classified_df['subject_prior'] >= 0.60].sort_values(
    'subject_prior', ascending=False)
print(f'HIGH-RISK TRENDING TOPICS (subject_prior ≥ 0.60): {len(high_risk)}')
print()
if len(high_risk):
    for _, row in high_risk.head(10).iterrows():
        print(f"  [{row['subject_prior']:.2f}] [{row['source']:10s}] {str(row['topic'])[:60]}")
        print(f"         → {row['category']}  (conf: {row.get('confidence', 0):.2f})")


## Step 5 — Compute trend amplifier
The trend amplifier modulates the static subject prior based on current attention velocity. High search velocity = higher probability of imminent synthetic amplification.

In [ ]:
# ── Trend amplifier ───────────────────────────────────────────────────────────

def compute_trend_amplifier(topic_classifications, decay_hours=48):
    """
    Compute a trend amplifier for each taxonomy category based on
    current trending data.
    
    Logic: If a topic in category X is currently trending with high velocity,
    the slop-factory demand signal for category X is elevated.
    
    The amplifier decays over time — trends from 48h ago carry less weight
    than current trends.
    
    Returns:
        amplifiers: dict mapping category -> amplifier value (1.0 = no amplification)
    """
    category_signals = {}
    
    for item in topic_classifications:
        cat = item.get('category', 'technical_educational')
        volume = item.get('volume', 0) or 0
        conf = item.get('confidence', 0.5)
        source = item.get('source', 'unknown')
        
        # Source weights (reliability and reach)
        source_weight = {'google': 1.0, 'reddit': 0.7, 'wikipedia': 0.6, 
                        'tiktok': 0.9, 'demo': 0.5}.get(source, 0.5)
        
        # Volume normalization (log scale)
        if volume > 0:
            vol_score = min(1.0, np.log1p(volume) / np.log1p(1_000_000))
        else:
            vol_score = 0.3  # present in trending but no volume data
        
        signal = vol_score * conf * source_weight
        
        if cat not in category_signals:
            category_signals[cat] = []
        category_signals[cat].append(signal)
    
    # Compute amplifier per category
    amplifiers = {}
    for cat in TAXONOMY_KEYS:
        signals = category_signals.get(cat, [])
        if signals:
            # Mean signal, boosted by presence of multiple trending items
            base_amp = np.mean(signals)
            count_boost = min(0.3, len(signals) * 0.1)
            amplifiers[cat] = 1.0 + base_amp + count_boost
        else:
            amplifiers[cat] = 1.0  # no boost if not trending
    
    return amplifiers


def compute_dynamic_prior(category_key, amplifiers):
    """
    Compute the dynamic subject prior by applying trend amplifier
    to the static taxonomy prior.
    
    P(slop | subject, trending) = min(1.0, P(slop|subject) × amplifier)
    """
    static_prior = compute_subject_prior(category_key)
    amp = amplifiers.get(category_key, 1.0)
    return min(1.0, static_prior * amp)


# Compute amplifiers from classified trending data
amplifiers = compute_trend_amplifier(classified)

# Build comparison table
amp_records = []
for cat in TAXONOMY_KEYS:
    static = compute_subject_prior(cat)
    dynamic = compute_dynamic_prior(cat, amplifiers)
    amp = amplifiers.get(cat, 1.0)
    amp_records.append({
        'category': cat.replace('_', ' '),
        'static_prior': round(static, 3),
        'amplifier': round(amp, 3),
        'dynamic_prior': round(dynamic, 3),
        'delta': round(dynamic - static, 3)
    })

amp_df = pd.DataFrame(amp_records).sort_values('dynamic_prior', ascending=False)

print('Dynamic Prior = Static Prior × Trend Amplifier')
print('═' * 65)
print(f'  {"Category":<30} {"Static":>7} {"Amp×":>7} {"Dynamic":>8} {"Δ":>7}')
print('  ' + '─' * 61)
for _, row in amp_df.iterrows():
    delta_str = f'+{row["delta"]:.3f}' if row['delta'] > 0 else f'{row["delta"]:.3f}'
    print(f'  {row["category"]:<30} {row["static_prior"]:>7.3f} '
          f'{row["amplifier"]:>7.2f}× {row["dynamic_prior"]:>8.3f} {delta_str:>7}')


In [ ]:
# ── Visualize static vs dynamic priors ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Subject Prior — Static vs Dynamic (Trend-Amplified)\n'
             'Dynamic prior incorporates current Google Trends / Reddit / Wikipedia signal',
             fontsize=12, fontweight='bold')

cats = amp_df['category'].tolist()
x = np.arange(len(cats))
width = 0.38

bars1 = axes[0].barh(x + width/2, amp_df['static_prior'], width,
                      label='Static prior', color='#4393c3', alpha=0.8)
bars2 = axes[0].barh(x - width/2, amp_df['dynamic_prior'], width,
                      label='Dynamic prior (trend-amplified)', color='#d6604d', alpha=0.8)

axes[0].set_yticks(x)
axes[0].set_yticklabels(cats, fontsize=7)
axes[0].set_xlabel('P(slop | subject)')
axes[0].set_title('Static vs Dynamic Subject Prior', fontsize=10)
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 1.0)
axes[0].axvline(0.7, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
axes[0].grid(axis='x', alpha=0.3)

# Delta chart — what the trend signal is changing
amp_df_sorted = amp_df.sort_values('delta', ascending=False)
colors_delta = ['#d62728' if d > 0.05 else '#aec7e8' for d in amp_df_sorted['delta']]
axes[1].barh(amp_df_sorted['category'], amp_df_sorted['delta'],
             color=colors_delta, alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Δ (Dynamic − Static prior)')
axes[1].set_title('Trend Amplification Effect\n(red = currently trending → elevated slop risk)',
                   fontsize=10)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].axvline(0.05, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
axes[1].grid(axis='x', alpha=0.3)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

print()
print('Categories with largest trend amplification RIGHT NOW:')
for _, row in amp_df_sorted[amp_df_sorted['delta'] > 0.02].head(5).iterrows():
    print(f'  +{row["delta"]:.3f}  {row["category"]}  '
          f'({row["static_prior"]:.3f} → {row["dynamic_prior"]:.3f})')


## Step 6 — Full SLURRY equation
Integrate all four axes into P(slop|content).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# THE SLURRY EQUATION
# ══════════════════════════════════════════════════════════════════════════════

def slurry_score(
    # Axis 1: Biological signal (from EVM)
    biological_snr=None,          # SNR in heartbeat band; None if not applicable
    evm_applicable=True,          # False for non-face/still content
    
    # Axis 2: Physical plausibility (from spectral analysis)
    physical_plausibility=None,   # composite score 0-1 from Axis 2
    
    # Axis 3: Subject prior (taxonomy + trend)
    subject_category=None,        # taxonomy key
    use_dynamic_prior=True,       # use trend-amplified prior
    amplifiers=None,              # pre-computed amplifiers dict
    
    # Axis 4: Context signals
    platform=None,                # 'tiktok','x','facebook','youtube','telegram',etc.
    account_age_days=None,        # account age in days
    post_velocity=None,           # posts per hour from account
    shares_in_first_hour=None,    # virality indicator
    election_proximity_days=None, # days to nearest election (None if not applicable
    
    # Priors
    base_rate=0.05,               # P(slop) in general content pool
    
    verbose=True
):
    """
    Compute SLURRY probability score: P(slop | content).
    
    This is NOT a binary classifier. It produces a probabilistic
    characterization across four axes with an interpretable composite.
    
    Returns:
        score: dict with per-axis scores and composite
    """
    scores = {}
    weights = {}
    
    # ── AXIS 1: Biological signal ──────────────────────────────────────────
    if not evm_applicable:
        scores['biological'] = None
        weights['biological'] = 0.0
    elif biological_snr is None:
        scores['biological'] = 0.5  # uncertain
        weights['biological'] = 0.15
    else:
        # High SNR in heartbeat band = real biological signal = lower slop prob
        # Low SNR = absent biological signal = higher slop prob
        # Sigmoid mapping: SNR of 0 → P(slop|bio)=0.85; SNR of 1 → 0.15
        bio_slop_prob = 1.0 / (1.0 + np.exp(4 * (biological_snr - 0.5)))
        scores['biological'] = round(float(bio_slop_prob), 3)
        weights['biological'] = 0.25
    
    # ── AXIS 2: Physical plausibility ─────────────────────────────────────
    if physical_plausibility is None:
        scores['physical'] = 0.5
        weights['physical'] = 0.15
    else:
        # High physical plausibility = real camera content = lower slop prob
        phys_slop_prob = 1.0 - physical_plausibility
        scores['physical'] = round(float(phys_slop_prob), 3)
        weights['physical'] = 0.25
    
    # ── AXIS 3: Subject prior ──────────────────────────────────────────────
    if subject_category is None:
        scores['subject'] = base_rate * 5  # mild elevation for unknown
        weights['subject'] = 0.20
    else:
        if use_dynamic_prior and amplifiers:
            subject_slop_prob = compute_dynamic_prior(subject_category, amplifiers)
        else:
            subject_slop_prob = compute_subject_prior(subject_category)
        scores['subject'] = round(float(subject_slop_prob), 3)
        weights['subject'] = 0.25
    
    # ── AXIS 4: Context signals ────────────────────────────────────────────
    context_signals = []
    
    # Platform risk
    platform_risk = {
        'telegram': 0.75, 'x': 0.65, 'tiktok': 0.60, 'facebook': 0.55,
        'whatsapp': 0.70, 'youtube': 0.45, 'instagram': 0.50,
        'news_site': 0.25, 'unknown': 0.50
    }.get((platform or 'unknown').lower(), 0.50)
    context_signals.append(platform_risk)
    
    # Account age (new accounts = higher slop risk)
    if account_age_days is not None:
        age_risk = max(0.1, 1.0 - min(1.0, account_age_days / 365))
        context_signals.append(age_risk)
    
    # Post velocity (bot-like behavior)
    if post_velocity is not None:
        vel_risk = min(1.0, post_velocity / 20)  # >20 posts/hr suspicious
        context_signals.append(vel_risk)
    
    # Virality (rapid shares = coordinated amplification)
    if shares_in_first_hour is not None:
        viral_risk = min(1.0, np.log1p(shares_in_first_hour) / np.log1p(10000))
        context_signals.append(viral_risk)
    
    # Election proximity
    if election_proximity_days is not None:
        elect_risk = max(0.0, 1.0 - election_proximity_days / 60)
        context_signals.append(elect_risk)
    
    context_score = float(np.mean(context_signals)) if context_signals else 0.5
    scores['context'] = round(context_score, 3)
    weights['context'] = 0.25 if any(v is not None for v in 
                                      [platform, account_age_days, post_velocity]) else 0.10
    
    # ── Normalize weights ──────────────────────────────────────────────────
    total_w = sum(weights.values())
    norm_weights = {k: v/total_w for k, v in weights.items()}
    
    # ── Composite score ────────────────────────────────────────────────────
    composite = 0.0
    for axis, score in scores.items():
        if score is not None:
            composite += score * norm_weights.get(axis, 0)
    
    # Bayesian-flavored correction: weight toward base_rate when evidence is weak
    evidence_strength = sum(1 for s in scores.values() if s is not None) / 4
    composite = (composite * evidence_strength + 
                 base_rate * (1 - evidence_strength))
    
    result = {
        'composite': round(composite, 3),
        'axis_scores': scores,
        'weights': norm_weights,
        'evidence_strength': round(evidence_strength, 2),
        'verdict': classify_composite(composite),
        'inputs': {
            'subject_category': subject_category,
            'platform': platform,
            'evm_applicable': evm_applicable,
        }
    }
    
    if verbose:
        print_slurry_report(result)
    
    return result


def classify_composite(score):
    """Map composite score to a SLURRY characterization — NOT a binary verdict."""
    if score >= 0.80: return 'HIGH SLOP PROBABILITY — strong epistemic risk signal'
    if score >= 0.65: return 'ELEVATED — multiple axes flagging, contextual investigation warranted'
    if score >= 0.50: return 'AMBIGUOUS — mixed signals, manual review recommended'
    if score >= 0.35: return 'LOW-MODERATE — some signals present, context dependent'
    return 'LOW — content consistent with authentic production'


def print_slurry_report(result):
    print('╔' + '═' * 55 + '╗')
    print('║  SLURRY PROTOCOL — Epistemic Risk Report' + ' ' * 14 + '║')
    print('╠' + '═' * 55 + '╣')
    print(f'║  Composite P(slop): {result["composite"]:.3f}' + ' ' * 34 + '║')
    print(f'║  {result["verdict"]}')
    print(f'║  Evidence strength: {result["evidence_strength"]:.2f}' + ' ' * 34 + '║')
    print('╠' + '═' * 55 + '╣')
    print('║  Axis Scores:' + ' ' * 41 + '║')
    for axis, score in result['axis_scores'].items():
        if score is not None:
            w = result['weights'].get(axis, 0)
            bar = '█' * int(score * 20)
            print(f'║    {axis:<12} {score:.3f}  {bar:<20}  (w={w:.2f})  ║')
        else:
            print(f'║    {axis:<12} N/A   (not applicable)' + ' ' * 16 + '║')
    print('╚' + '═' * 55 + '╝')


print('SLURRY equation loaded.')
print()
print('Usage:')
print('  result = slurry_score(')
print('      biological_snr=0.12,           # from EVM analysis')
print('      physical_plausibility=0.28,    # from spectral analysis')
print('      subject_category="political_figure_polarizing",')
print('      platform="x",')
print('      election_proximity_days=14,')
print('      amplifiers=amplifiers          # from trend signal')
print('  )')


In [ ]:
# ── Worked examples — GMF catalog cases ──────────────────────────────────────

print('Running SLURRY equation on GMF documented cases...')
print()

GMF_SCENARIOS = [
    {
        'name': 'Slovakia Audio (2023)',
        'biological_snr': None,        # audio only
        'evm_applicable': False,
        'physical_plausibility': None,  # audio — spectral different analysis needed
        'subject_category': 'electoral_process',
        'platform': 'facebook',
        'account_age_days': 30,
        'election_proximity_days': 3,
        'shares_in_first_hour': 5000,
    },
    {
        'name': 'Biden NH Robocall (2024)',
        'biological_snr': None,
        'evm_applicable': False,
        'physical_plausibility': None,
        'subject_category': 'political_figure_polarizing',
        'platform': 'unknown',  # phone
        'election_proximity_days': 7,
    },
    {
        'name': 'Taylor Swift AI Images (2024)',
        'biological_snr': 0.05,        # static images — no temporal signal
        'evm_applicable': False,
        'physical_plausibility': 0.22,  # AI-generated — low physical plausibility
        'subject_category': 'celebrity_political_endorsement',
        'platform': 'x',
        'shares_in_first_hour': 50000,
        'account_age_days': 2,
    },
    {
        'name': 'Imran Khan Prison Video (2024)',
        'biological_snr': 0.08,        # AI video — very low biological signal
        'evm_applicable': True,
        'physical_plausibility': 0.31,
        'subject_category': 'political_figure_polarizing',
        'platform': 'youtube',
        'election_proximity_days': 10,
        'shares_in_first_hour': 20000,
    },
    {
        'name': 'Springfield Pets Images (2024)',
        'biological_snr': 0.04,
        'evm_applicable': False,        # still images
        'physical_plausibility': 0.18,  # diffusion-generated — low plausibility
        'subject_category': 'cute_animals_wholesome',  # vessel risk paradigm case
        'platform': 'x',
        'shares_in_first_hour': 100000,
        'election_proximity_days': 58,
    },
    {
        'name': 'Authentic Press Conference (control)',
        'biological_snr': 0.72,        # real video — strong biological signal
        'evm_applicable': True,
        'physical_plausibility': 0.84,  # real camera
        'subject_category': 'political_figure_mainstream',
        'platform': 'youtube',
        'account_age_days': 2500,
        'shares_in_first_hour': 200,
    },
]

results = []
for scenario in GMF_SCENARIOS:
    print(f'── {scenario["name"]} ' + '─' * (45 - len(scenario['name'])))
    r = slurry_score(
        biological_snr=scenario.get('biological_snr'),
        evm_applicable=scenario.get('evm_applicable', True),
        physical_plausibility=scenario.get('physical_plausibility'),
        subject_category=scenario.get('subject_category'),
        platform=scenario.get('platform'),
        account_age_days=scenario.get('account_age_days'),
        shares_in_first_hour=scenario.get('shares_in_first_hour'),
        election_proximity_days=scenario.get('election_proximity_days'),
        amplifiers=amplifiers,
        verbose=True
    )
    r['name'] = scenario['name']
    results.append(r)
    print()


In [ ]:
# ── Comparative visualization ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SLURRY Score — GMF Documented Cases + Control\n'
             'P(slop | content) across four axes',
             fontsize=13, fontweight='bold')

names = [r['name'] for r in results]
composites = [r['composite'] for r in results]
axis_labels = ['biological', 'physical', 'subject', 'context']
axis_colors = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

# Composite bar chart
bar_colors = ['#d62728' if s >= 0.65 else '#ff7f0e' if s >= 0.50
              else '#2ca02c' for s in composites]
bars = axes[0].barh(names, composites, color=bar_colors, alpha=0.85,
                    edgecolor='white', linewidth=0.8)
axes[0].set_xlabel('P(slop | content) — Composite Score')
axes[0].set_title('Composite SLURRY Score', fontsize=10)
axes[0].set_xlim(0, 1.0)
axes[0].axvline(0.65, color='red', linestyle='--', alpha=0.5,
                linewidth=1.0, label='Elevated threshold')
axes[0].axvline(0.50, color='orange', linestyle='--', alpha=0.5,
                linewidth=1.0, label='Ambiguous threshold')
axes[0].legend(fontsize=8)
axes[0].grid(axis='x', alpha=0.3)
for bar, score in zip(bars, composites):
    axes[0].text(score + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=9, fontweight='bold')

# Stacked axis breakdown
x = np.arange(len(names))
width = 0.15
for i, (ax_label, color) in enumerate(zip(axis_labels, axis_colors)):
    vals = []
    for r in results:
        v = r['axis_scores'].get(ax_label)
        vals.append(v if v is not None else 0.0)
    axes[1].bar(x + i * width, vals, width, label=ax_label.capitalize(),
                color=color, alpha=0.8, edgecolor='white')

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels([n.split('(')[0].strip() for n in names],
                         rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Axis Score (0=authentic, 1=slop)')
axes[1].set_title('Per-Axis Breakdown', fontsize=10)
axes[1].legend(fontsize=8)
axes[1].set_ylim(0, 1.0)
axes[1].axhline(0.65, color='red', linestyle='--', alpha=0.3, linewidth=0.8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


---
## SLURRY Protocol — Complete Architecture

```
                    ┌─────────────────────────────────────────────┐
                    │           SLURRY PROTOCOL v0.1              │
                    │   Synthetic Likelihood Using Residual        │
                    │   and Real-world Yield                       │
                    └──────────────────┬──────────────────────────┘
                                       │
              ┌────────────────────────┼────────────────────────┐
              │                        │                        │
    ┌─────────▼──────┐     ┌──────────▼──────┐     ┌──────────▼──────┐
    │    AXIS 1      │     │    AXIS 2        │     │    AXIS 3       │
    │  Biological    │     │  Physical        │     │  Subject        │
    │  Signal (EVM)  │     │  Plausibility    │     │  Prior          │
    │                │     │  (Spectral)      │     │                 │
    │ · Blood flow   │     │ · PRNU noise     │     │ · Taxonomy      │
    │ · Heart band   │     │ · FFT artifacts  │     │ · Trend amp     │
    │ · Spatial map  │     │ · Chrom. aber.   │     │ · Vessel risk   │
    │                │     │ · Bayer stats    │     │ · Google/Reddit │
    └────────┬───────┘     └─────────┬────────┘     └────────┬────────┘
             │                       │                       │
             └───────────────────────┴───────────────────────┘
                                     │
                           ┌─────────▼────────┐
                           │    AXIS 4        │
                           │  Context Prior   │
                           │                  │
                           │ · Platform risk  │
                           │ · Account age    │
                           │ · Virality       │
                           │ · Election prox. │
                           └─────────┬────────┘
                                     │
                    ┌────────────────▼─────────────────┐
                    │     P(slop | content)             │
                    │                                   │
                    │  NOT a binary verdict —           │
                    │  a multi-dimensional              │
                    │  epistemic risk characterization  │
                    └───────────────────────────────────┘
```

### What makes this SLURRY and not forensics

Forensic tools produce verdicts on individual artifacts — real or fake, pass or fail.  
SLURRY produces a **characterization profile** that asks:

1. Does this content carry biological traces of having been *lived*? (Axis 1)
2. Does this content carry physical traces of having *existed*? (Axis 2)  
3. Is this content about a subject that *attracts* synthetic manipulation? (Axis 3)
4. Is this content *deployed* in a context associated with synthetic amplification? (Axis 4)

The epistemic question is not *"is this fake?"* but *"what does this content do 
to the information environment, and what does it carry ideologically?"*

---
*SLURRY Protocol v0.1 — Full four-axis implementation*  
*Jon Nealon, DMI Summer School 2026, University of Amsterdam*  
*Framework: SLURRY (Synthetic Likelihood Using Residual and Real-world Yield)*
